## HyperParams :
* light df vs real df
* model dino2 vs dino 3
* metadata
* extracted labels
* num_slices
* SSL
* model architecture , specific target view Routing


TO DO
step 1:
* Target view Routing
step 2:
* Real DF instead of Light : feed multiple plane for each Orientation


 v11 : Benchmark 
* No Metadata
* dinov2_small
* LIGHT_DF instead of total merged_series
* NUM_SLICES = 16
* IMAGE_SIZE = 256
* BATCH_SIZE = 16
* ACCUMULATION_STEP = 4
* WARMUP_EPOCHS = 0
* PATIENCE = 7

 v16 :
* SSL training on dicom images sim-siam

 v18 : Benchmark2 (using SSL pretrained model)
* No Metadata
* dinov2_small
* LIGHT_DF instead of total merged_series
* NUM_SLICES = 16
* IMAGE_SIZE = 224
* BATCH_SIZE = 32
* ACCUMULATION_STEP = 4
* WARMUP_EPOCHS = 0
* PATIENCE = 9

 20 :
* SSL training on dicom images light teacher-student

 v33 : resume v25 (using SSL pretrained model)
* No Metadata
* dinov2_small
* LIGHT_DF instead of total merged_series
* NUM_SLICES = 16
* IMAGE_SIZE = 256
* BATCH_SIZE = 8
* encoder unfreezed after cycle 1
* ACCUMULATION_STEP = 8
* WARMUP_EPOCHS = 0
* PATIENCE = 9

 v36 : SSL Training with fixes

 v37 : LB: 0.79
* first 2 epochs frozen
* gradually unfreezing dinov2 blocks
* pr resizing after 12 epoch 256==>384

 v38 : Resume 37


 v72
 * metadata in meta encoder 
 * img 256
 * light df
 * view routing
 * 32 meta feature vector ,256 image feature vector
 ------------------
 * dropout : 0.3
 * ssl on full data

v73 -75
* metadata ==> lightGBM
* images ===>  2.5d architecture
* image model macro Auc + w (meta macro auc)
* two weights appoach: 1- one weihgts for all label (meta model) 2- 12 weights for each label(meta model)
* new labels extracted from reports : macro auc: 80==>85
* ssl on light df

v78
pr resizing to 384 after 16 cycle

v 8-81
convnext ssl on full data



In [1]:
!pip install /kaggle/input/datasets/malihealavi/monai-1-4-0/monai-1.4.0-py3-none-any.whl \
    --no-deps \
    --force-reinstall

from IPython.display import clear_output
clear_output()
import monai
print(monai.__version__)  # Should show 1.4.0

<frozen importlib._bootstrap_external>:1301: FutureWarning: The cuda.cudart module is deprecated and will be removed in a future release, please switch to use the cuda.bindings.runtime module instead.


1.4.0


In [2]:
import warnings
warnings.filterwarnings("ignore")
import os
os.environ["NCCL_P2P_DISABLE"] = "1"
os.environ["NCCL_IB_DISABLE"] = "1"
import sys 
import json
import ast
from glob import glob
import random
import collections
import time
import numpy as np
import re
import gc
import monai.transforms as mt
from typing import Optional
from typing import List
import timm

import numpy as np
import pandas as pd
import pydicom
from pydicom.pixel_data_handlers.util import apply_voi_lut
import cv2
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import roc_auc_score
from tqdm import tqdm
from transformers import AutoImageProcessor, AutoModel

from lightgbm import LGBMClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from scipy.special import logit, expit
from sklearn.metrics import roc_auc_score
from sklearn.ensemble import HistGradientBoostingClassifier
from scipy.optimize import minimize
from sklearn.model_selection import KFold

 
import torch
from torch import nn
from torch.utils import data as torch_data
from sklearn import model_selection as sk_model_selection
from torch.nn import functional as torch_functional
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torch.optim as optim
from torch.optim.lr_scheduler import ReduceLROnPlateau , LinearLR, CosineAnnealingLR, SequentialLR
from torch.cuda.amp import autocast, GradScaler

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.preprocessing import StandardScaler
from IPython.display import clear_output
clear_output()
            
import re

  

# Global Variable 

In [3]:
TRAIN_SERIES = pd.read_csv('/kaggle/input/competitions/rsna-knee-abnormality-detection/train_series.csv')
TEST_SERIES = pd.read_csv('/kaggle/input/competitions/rsna-knee-abnormality-detection/test_series.csv')
SERIES_MERGED = pd.read_csv('/kaggle/input/knee-abnormally-dataset/df_merged.csv')
# TRAIN_LABELD = pd.read_csv('/kaggle/input/knee-abnormally-dataset/labeld_df.csv')
TRAIN_LABELD_OLD = pd.read_csv('/kaggle/input/knee-abnormally-dataset/labeld_df.csv')
TRAIN_LABELD = pd.read_csv('/kaggle/input/knee-abnormally-dataset/labeled_df_by_group.csv')
TRAIN = pd.read_csv('/kaggle/input/competitions/rsna-knee-abnormality-detection/train.csv')
TRAIN_LABELD [['Medial OA', 'Lateral OA']] = TRAIN_LABELD_OLD[['Medial OA', 'Lateral OA']]

TRUE_LABELS_DF = TRAIN[TRAIN.notna().all(axis=1)]
GT_VAL = TRAIN_LABELD.iloc[TRUE_LABELS_DF.index]
TRAIN_LABELD = TRAIN_LABELD.drop(TRUE_LABELS_DF.index)

LABELS= ['ACL', 'MCL', 'Medial Meniscus', 'Lateral Meniscus', 'Medial OA',
         'Lateral OA', 'PF OA', 'Effusion', 'Synovitis', "Baker's", "Contusion", 'Fracture']

NUM_SLICES = 16
IMAGE_SIZE = 256
BATCH_SIZE = 8
ACCUMULATION_STEP = 4
WARMUP_EPOCHS = 0
MIN_DELTA = 0.001   # early stopping if auc improvement < min_delta fo 6 epochs
PATIENCE = 6
ENCODER_BB = 'convnext_nano'
NUM_CYCLE= 30
NUM_CHUNKS= 3

TOTAL_BACKBONE_BLOCKS = 12   # dinov2_small has 12 transformer blocks (confirmed from your SSL run output)
UNFREEZE_START_CYCLE  = 2    # backbone stays fully frozen before this cycle
CYCLES_PER_BLOCK      = 1    # +1 more block unfrozen every this many cycles
MAX_UNFROZEN_BLOCKS   = 4    # cap — matches the "last 1-2 blocks" target from before
TOTAL_CONV_STAGES = 2     

In [4]:
# pick series with atleast either Fluid_Sensitive == True or Fat_Suppression == True 
import pandas as pd 
def pick_best_series(group):
    best = group[(group.Fluid_Sensitive == True) & (group.Fat_Suppression == True)]
    if len(best) == 0:
        best = group[group.Fluid_Sensitive == True]   # fallback: fluid-sensitive only
    if len(best) == 0:
        best = group                                  # fallback: take whatever exists
    return best.iloc[0]
complete_series_merged = SERIES_MERGED.copy()
LIGHT_DF = (
    complete_series_merged
    .groupby(['StudyInstanceUID', 'Anatomical_Plane'])
    .apply(pick_best_series)
    .reset_index(drop=True)
)
LIGHT_DF

,StudyInstanceUID,SeriesInstanceUID,Fluid_Sensitive,Fat_Suppression,Anatomical_Plane,file_path,InstanceNumber,SeriesDescription,BodyPartExamined,Laterality,PatientSex,ScanningSequence,SequenceVariant,ScanOptions,MRAcquisitionType,Rows,Columns,MagneticFieldStrength,FlipAngle,Manufacturer
0,1.2.826.0.1.3680043.8.498.10004873229099053869...,1.2.826.0.1.3680043.8.498.13821229744997220641...,1,1,Axial,['/kaggle/input/competitions/rsna-knee-abnorma...,18,STIR CS_ AX,KNEE,NaN,F,IR,SK,OTHER,2D,560,560,1.5,90.0,Philips
1,1.2.826.0.1.3680043.8.498.10004873229099053869...,1.2.826.0.1.3680043.8.498.40734206102458723096...,1,1,Coronal,['/kaggle/input/competitions/rsna-knee-abnorma...,16,DP SPAIR_CS,KNEE,NaN,F,SE,SK,FS,2D,560,560,1.5,90.0,Philips
2,1.2.826.0.1.3680043.8.498.10004873229099053869...,1.2.826.0.1.3680043.8.498.12343110195036213483...,1,1,Sagittal,['/kaggle/input/competitions/rsna-knee-abnorma...,22,DP SPIR CS_SAG,KNEE,NaN,F,SE,SK,FS,2D,512,512,1.5,90.0,Philips
3,1.2.826.0.1.3680043.8.498.10004945927472656027...,1.2.826.0.1.3680043.8.498.83327574802189551711...,1,1,Axial,['/kaggle/input/competitions/rsna-knee-abnorma...,38,pd_tse_fs_tra_448_p4_s2_DRB,KNEE,RIGHT,M,SE,"['SP', 'SK']","['SP', 'FS', 'PER']",2D,640,640,3.0,180.0,Siemens
4,1.2.826.0.1.3680043.8.498.10004945927472656027...,1.2.826.0.1.3680043.8.498.99023014295026495734...,1,1,Coronal,['/kaggle/input/competitions/rsna-knee-abnorma...,30,pd_tse_fs_cor_320_p4_s2_DRB,KNEE,RIGHT,M,SE,"['SP', 'SK']","['FS', 'PER']",2D,640,540,3.0,150.0,Siemens
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
13216,1.2.826.0.1.3680043.8.498.99939100657291954890...,1.2.826.0.1.3680043.8.498.75652438013374920566...,1,1,Coronal,['/kaggle/input/competitions/rsna-knee-abnorma...,24,CO T2W FS,EXTREMITY,NaN,NaN,NaN,NaN,NaN,NaN,512,512,NaN,NaN,Toshiba
13217,1.2.826.0.1.3680043.8.498.99939100657291954890...,1.2.826.0.1.3680043.8.498.54980506759743668422...,1,1,Sagittal,['/kaggle/input/competitions/rsna-knee-abnorma...,24,SG T2W FS,EXTREMITY,NaN,NaN,NaN,NaN,NaN,NaN,512,512,NaN,NaN,Toshiba
13218,1.2.826.0.1.3680043.8.498.99957611207313098696...,1.2.826.0.1.3680043.8.498.54309001783901668483...,1,1,Axial,['/kaggle/input/competitions/rsna-knee-abnorma...,20,AXI DP SPIR,KNEE,NaN,M,SE,SK,FS,2D,512,512,1.5,90.0,Philips
13219,1.2.826.0.1.3680043.8.498.99957611207313098696...,1.2.826.0.1.3680043.8.498.62870298129639401344...,1,1,Coronal,['/kaggle/input/competitions/rsna-knee-abnorma...,21,CORONAL DP SPAIR,KNEE,NaN,M,SE,SK,FS,2D,560,560,1.5,90.0,Philips


In [5]:
def seed_everything(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    # Warning: the following lines can slow down training performance slightly
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything()

# feature Engineering 

In [6]:
cols = ['InstanceNumber', 'Laterality', 'PatientSex',
       'ScanningSequence', 'SequenceVariant', 
       'MRAcquisitionType', 'Rows', 'Columns', 'MagneticFieldStrength',
       'FlipAngle', 'Manufacturer', ]
for c in cols:
    print(f'{c} : {SERIES_MERGED[c].unique()}')

InstanceNumber : [ 22  18  16  34  38  30  48  32  29 160  40  35  41  51  20  27  24  26
  36  23  19  21  15  17  31  25 320 186  60  33  28 132 120  39  45  11
  37  42 144 136  53  70  14  46 164  43 128 200  50 130 100  72  13  62
  52  12  80 152  88 192  82  44  49  76 140  79  87  47  84  90 134  54
 176 150  78  57  66 106  55 208 254 104  68 148  86  64 116 124  92  65
  74  69 126 112 172  58  96  63  56  67]
Laterality : [nan 'RIGHT' 'LEFT' 'B']
PatientSex : ['F' 'M' nan 'O']
ScanningSequence : ['SE' 'IR' 'GR' nan 'RM' "['SE', 'IR']" "['IR', 'SE']"]
SequenceVariant : ['SK' "['SP', 'SK']" "['SK', 'SP', 'OSP']" "['SP', 'OSP']" "['SK', 'OSP']"
 nan "['SK', 'SP']" 'SP' "['SS', 'SK']" 'NONE' 'OSP' "['OSP', 'WI']"
 "['IR', 'OSP']" 'SS' "['SS', 'OSP']" "['SK', 'SP', 'MP', 'OSP']"
 "['SS', 'SK', 'OSP']"]
MRAcquisitionType : ['2D' '3D' nan]
Rows : [ 512  560  576  640  768  416  384  256  320 1024  672  800  704  496
  448  464  896  792  720  528  740  564  540  480  432  400  352 

In [7]:
import ast
import pandas as pd
import numpy as np

def clean_and_encode_dicom_metadata(df: pd.DataFrame) -> pd.DataFrame:
    """
    Cleans, aggregates, and One-Hot Encodes categorical DICOM metadata columns.
    
    Parameters:
    -----------
    df : pd.DataFrame
        DataFrame containing columns: Manufacturer, Laterality, PatientSex,
        ScanningSequence, SequenceVariant, MRAcquisitionType
        
    Returns:
    --------
    df_encoded : pd.DataFrame
        DataFrame with original metadata columns cleaned and One-Hot Encoded.
    """
    df = df.copy()
    
    # -------------------------------------------------------------------------
    # 1. Clean & Aggregate Manufacturer
    # -------------------------------------------------------------------------

    df['Manufacturer_Clean'] = df['Manufacturer'].copy()

    # -------------------------------------------------------------------------
    # 2. Clean & Aggregate Laterality
    # -------------------------------------------------------------------------
    def clean_laterality(val):
        if pd.isna(val) or str(val).strip() == '':
            return 'Unknown'
        val = str(val).strip().upper()
        mapping = {
            'R': 'RIGHT',
            'RIGHT': 'RIGHT',
            'L': 'LEFT',
            'LEFT': 'LEFT',
            'B': 'BILATERAL',
            'BILATERAL': 'BILATERAL'
        }
        return mapping.get(val, 'Unknown')

    df['Laterality_Clean'] = df['Laterality'].apply(clean_laterality)

    # -------------------------------------------------------------------------
    # 3. Clean & Aggregate PatientSex
    # -------------------------------------------------------------------------
    def clean_patient_sex(val):
        if pd.isna(val):
            return 'Undefined'
        val = str(val).strip().upper()
        if val == 'M':
            return 'Male'
        elif val == 'F':
            return 'Female'
        return 'Undefined'

    df['PatientSex_Clean'] = df['PatientSex'].apply(clean_patient_sex)

    # -------------------------------------------------------------------------
    # 4. Clean & Aggregate ScanningSequence
    # -------------------------------------------------------------------------
    def parse_sequence_list(val):
        if pd.isna(val):
            return []
        val_str = str(val).strip()
        # Parse string representation of list (e.g. "['SE', 'IR']")
        if val_str.startswith('['):
            try:
                parsed = ast.literal_eval(val_str)
                if isinstance(parsed, list):
                    return [str(x).strip() for x in parsed]
            except (ValueError, SyntaxError):
                pass
        return [val_str] if val_str else []
    
    df['ScanningSequence_Clean'] = df['ScanningSequence'].apply(parse_sequence_list)
    # Binary column for each sequence type (1 if present in the list)
    for seq in ['SE', 'GR', 'IR', 'RM']:
        df[f'ScanningSequence_{seq}'] = df['ScanningSequence_Clean'].apply(
            lambda x: int(seq in x)
        ).astype('int8')
    df.drop(columns=['ScanningSequence_Clean'], inplace=True)

    # -------------------------------------------------------------------------
    # 5. Clean & Aggregate SequenceVariant
    # -------------------------------------------------------------------------
    def parse_variant_list(val):
        if pd.isna(val):
            return []
        val_str = str(val).strip()
        if val_str.startswith('['):
            try:
                parsed = ast.literal_eval(val_str)
                if isinstance(parsed, list):
                    return [str(x).strip() for x in parsed if str(x).strip() != 'NONE']
            except (ValueError, SyntaxError):
                pass
        return [val_str] if val_str not in ('', 'NONE') else []

    df['SequenceVariant_Clean'] = df['SequenceVariant'].apply(parse_variant_list)
    # Presence flags for each variant type
    for var in ['SK', 'SP', 'OSP', 'SS']:
        df[f'SequenceVariant_{var}'] = df['SequenceVariant_Clean'].apply(
            lambda x: int(var in x)
        ).astype('int8')
    df.drop(columns=['SequenceVariant_Clean'], inplace=True)

    # -------------------------------------------------------------------------
    # 6. Clean & Aggregate MRAcquisitionType
    # -------------------------------------------------------------------------
    def clean_acq_type(val):
        if pd.isna(val) or str(val).strip() == '':
            return 'UNKNOWN'
        val_str = str(val).strip().upper()
        return val_str if val_str in ['2D', '3D'] else 'UNKNOWN'

    df['MRAcquisitionType_Clean'] = df['MRAcquisitionType'].apply(clean_acq_type)

    # -------------------------------------------------------------------------
    # 7. One-Hot Encoding
    # -------------------------------------------------------------------------
    clean_cols = [
        'Manufacturer_Clean',
        'Laterality_Clean',
        'PatientSex_Clean',
        # 'ScanningSequence_Clean',
        # 'SequenceVariant_Clean',
        'MRAcquisitionType_Clean'
    ]
    
    # Perform One-Hot Encoding and convert boolean outputs to integers (0 or 1)
    df_encoded = pd.get_dummies(df, columns=clean_cols, prefix_sep='_', dtype=int)
    
    # Drop original uncleaned raw columns
    raw_cols = ['Manufacturer', 'Laterality', 'PatientSex', 'ScanningSequence', 'SequenceVariant', 'MRAcquisitionType']
    df_encoded = df_encoded.drop(columns=[col for col in raw_cols if col in df_encoded.columns])

    return df_encoded

sample_df = LIGHT_DF[['Manufacturer',
                'Laterality',
                'PatientSex',
                 'ScanningSequence',
                'SequenceVariant',
                'MRAcquisitionType']].copy()
# Run cleaning and encoding
encoded_df = clean_and_encode_dicom_metadata(sample_df)
encoded_df['SliceNumberRate'] = NUM_SLICES/LIGHT_DF['InstanceNumber']
encoded_df['InstanceNumber'] = LIGHT_DF['InstanceNumber']
encoded_df['Rows'] = np.log(LIGHT_DF['Rows'])
encoded_df['Columns'] = np.log(LIGHT_DF['Columns'])
encoded_df[[ 'MagneticFieldStrength','FlipAngle' , 'Fluid_Sensitive', 'Fat_Suppression'	]] = LIGHT_DF[[ 'MagneticFieldStrength','FlipAngle' , 'Fluid_Sensitive', 'Fat_Suppression'	]]
print("Encoded Feature Columns:")
print(encoded_df.columns.tolist())


num_cols = ['InstanceNumber', 'Rows', 'Columns', 'MagneticFieldStrength',
            'FlipAngle', 'SliceNumberRate']

std_scaler = StandardScaler()
encoded_df[num_cols] = std_scaler.fit_transform(encoded_df[num_cols])

# Impute AFTER scaling so missing becomes an out-of-range sentinel
encoded_df[num_cols] = encoded_df[num_cols].fillna(-1)
encoded_df [['StudyInstanceUID','SeriesInstanceUID','file_path', 'Anatomical_Plane']] = LIGHT_DF[['StudyInstanceUID','SeriesInstanceUID','file_path', 'Anatomical_Plane']]
LIGHT_DF = None
LIGHT_DF = encoded_df.copy()
LIGHT_DF.info()

Encoded Feature Columns:
['ScanningSequence_SE', 'ScanningSequence_GR', 'ScanningSequence_IR', 'ScanningSequence_RM', 'SequenceVariant_SK', 'SequenceVariant_SP', 'SequenceVariant_OSP', 'SequenceVariant_SS', 'Manufacturer_Clean_Canon', 'Manufacturer_Clean_Fujifilm', 'Manufacturer_Clean_GE', 'Manufacturer_Clean_Hitachi', 'Manufacturer_Clean_Philips', 'Manufacturer_Clean_Siemens', 'Manufacturer_Clean_Toshiba', 'Laterality_Clean_BILATERAL', 'Laterality_Clean_LEFT', 'Laterality_Clean_RIGHT', 'Laterality_Clean_Unknown', 'PatientSex_Clean_Female', 'PatientSex_Clean_Male', 'PatientSex_Clean_Undefined', 'MRAcquisitionType_Clean_2D', 'MRAcquisitionType_Clean_3D', 'MRAcquisitionType_Clean_UNKNOWN', 'SliceNumberRate', 'InstanceNumber', 'Rows', 'Columns', 'MagneticFieldStrength', 'FlipAngle', 'Fluid_Sensitive', 'Fat_Suppression']
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 13221 entries, 0 to 13220
Data columns (total 37 columns):
 #   Column                           Non-Null Count  Dtype  


# Aug and Dataset

In [8]:

def get_base_transform(num_slices, image_size, mri_keys):
    # Always runs: resize + normalize. Needed for every sample, aug or not.
    return mt.Compose([
        mt.EnsureChannelFirstd(keys=mri_keys, channel_dim="no_channel"),
        mt.Resized(keys=mri_keys, spatial_size=(num_slices, image_size, image_size), mode="trilinear"),
        mt.NormalizeIntensityd(keys=mri_keys, nonzero=True, channel_wise=True),
    ])
    
def get_rand_transform(mri_keys):
    # Only runs for training when aug=True. No resize/normalize here — base already did it.
    return mt.Compose([
        mt.RandFlipd(keys=mri_keys, prob=0.5, spatial_axis=2),
        mt.RandFlipd(keys=mri_keys, prob=0.5, spatial_axis=1),
        mt.RandRotated(
            keys=mri_keys,
            range_x=0.2, range_y=0.2, range_z=0.2,
            prob=0.5, mode="bilinear", padding_mode="zeros"
        ),
        mt.RandZoomd(keys=mri_keys, min_zoom=0.9, max_zoom=1.1, prob=0.5),
        mt.RandGaussianNoised(keys=mri_keys, prob=0.3, mean=0.0, std=0.05),
        mt.RandShiftIntensityd(keys=mri_keys, offsets=0.1, prob=0.4),
        mt.RandScaleIntensityd(keys=mri_keys, factors=0.15, prob=0.4),
        mt.RandBiasFieldd(keys=mri_keys, prob=0.3, coeff_range=(0.0, 0.3)),
        mt.RandGibbsNoised(keys=mri_keys, prob=0.15, alpha=(0.0, 0.4)),
    ])

    

def load_dicom_image(path, img_size, voi_lut=True, rotate=0):
    try:
        dicom = pydicom.dcmread(path)
        
        # Apply VOI LUT transformation if requested and available
        if voi_lut:
            try:
                data = apply_voi_lut(dicom.pixel_array, dicom)
            except Exception:
                data = dicom.pixel_array
        else:
            data = dicom.pixel_array
            
        # Standardize pixel output data type
        data = data.astype(np.float32)
        
        # Apply rotation if specified
        if rotate != 0:
            # Rotates matrix in 90-degree increments
            k = rotate // 90
            data = np.rot90(data, k=k)
            
    except Exception:
        return np.zeros((img_size, img_size), dtype=np.float32)
        
    return data



def load_dicom_images_3d(files, num_imgs, img_size, rotate=0, data_directory="."):
    if len(files) == 0:
        return np.zeros((num_imgs, img_size, img_size), dtype=np.float32)

    middle = len(files) // 2
    half = num_imgs // 2
    p1, p2 = max(0, middle - half), min(len(files), middle + half)

    imgs = []
    for f in files[p1:p2]:
        im = load_dicom_image(f, img_size=img_size, rotate=rotate, voi_lut=True)
        im = np.nan_to_num(np.asarray(im, dtype=np.float32), nan=0.0, posinf=0.0, neginf=0.0)
        if im.ndim != 2:                       # multi-frame / RGB files
            im = np.zeros((img_size, img_size), dtype=np.float32)
        imgs.append(im)

    if not imgs:
        return np.zeros((num_imgs, img_size, img_size), dtype=np.float32)

    # use the most common slice shape; replace slices with another shape by zeros
    shapes = [im.shape for im in imgs]
    target = max(set(shapes), key=shapes.count)
    imgs = [im if im.shape == target else np.zeros(target, dtype=np.float32) for im in imgs]
    return np.stack(imgs, axis=0)



In [9]:

def stack_neighbors(volume: torch.Tensor, window: int = 1) -> torch.Tensor:
    """
    volume: (N, H, W)  — single modality, N slices
    returns: (N, 2*window+1, H, W)  — each slice paired with neighbors
    """
    N, H, W = volume.shape
    slices = []
    for i in range(N):
        neighbors = []
        for offset in range(-window, window + 1):
            idx = max(0, min(N - 1, i + offset))
            neighbors.append(volume[idx])
        slices.append(torch.stack(neighbors, dim=0))  # (3, H, W)
    return torch.stack(slices, dim=0)



class KneeDataset(Dataset):
    PLANES = ['Axial', 'Coronal', 'Sagittal']
    TARGETS = ['ACL', 'MCL', 'Medial Meniscus', 'Lateral Meniscus', 'Medial OA',
               'Lateral OA', 'PF OA', 'Effusion', 'Synovitis', "Baker's", "Contusion", 'Fracture']

    def __init__(self, train_labeled: pd.DataFrame = TRAIN_LABELD, series_merged: pd.DataFrame = LIGHT_DF,
                 split: str = 'train', augmentation=True, stack_neighbor=True, meta_ref=None):
        """meta_ref = (meta_cols, mean, std) from the training dataset (use train_ds.meta_ref at test time)."""
        self.aug = augmentation
        self.split = split
        self.stack_neighbor = stack_neighbor
        self.train_labeled_df = train_labeled.to_dict(orient='records')
        self.series_merged_df = series_merged.set_index('SeriesInstanceUID')
        self.groupby_studyUID = series_merged.groupby('StudyInstanceUID').agg({
            'Anatomical_Plane': list,
            'SeriesInstanceUID': list
        })

        exclude_cols = ['StudyInstanceUID', 'SeriesInstanceUID', 'file_path', 'Anatomical_Plane', 'Report']
        if meta_ref is not None:
            self.meta_cols = list(meta_ref[0])          # same columns, same order as training
        else:
            self.meta_cols = [c for c in self.series_merged_df.columns
                              if c not in exclude_cols + self.TARGETS]

       
        self.num_meta = len(self.meta_cols)

    def __len__(self):
        return len(self.train_labeled_df)

    def _zeros_planes(self):
        d = {}
        for p in self.PLANES:
            t = torch.zeros((NUM_SLICES, IMAGE_SIZE, IMAGE_SIZE), dtype=torch.float32)
            d[p] = stack_neighbors(t, window=1) if self.stack_neighbor else t
        return d

    def __getitem__(self, idx):
        row = self.train_labeled_df[idx]
        plane_order = self.PLANES
        meta_dict = {p: np.zeros(self.num_meta, dtype=np.float32) for p in plane_order}
        dict_3d = {}

        try:
            gp_row = self.groupby_studyUID.loc[row['StudyInstanceUID']]

            for i, mri_type in enumerate(gp_row['Anatomical_Plane']):
                if mri_type not in plane_order:          # NaN / unknown plane name
                    continue
                try:                                      # one bad plane must not kill the others
                    uid = gp_row['SeriesInstanceUID'][i]
                    files = ast.literal_eval(self.series_merged_df.loc[uid]['file_path'])

                    if len(files) == 0:
                        dict_3d[mri_type] = torch.zeros((NUM_SLICES, IMAGE_SIZE, IMAGE_SIZE), dtype=torch.float32)
                        continue

                    vol = load_dicom_images_3d(files, num_imgs=NUM_SLICES, img_size=IMAGE_SIZE)
                    base = get_base_transform(num_slices=NUM_SLICES, image_size=IMAGE_SIZE, mri_keys=[mri_type])
                    dict_3d[mri_type] = base({mri_type: vol})[mri_type]
            
                except Exception as e:
                    print(f"[SKIP plane] {row['StudyInstanceUID']} {mri_type}: {e}")
                    dict_3d[mri_type] = torch.zeros((NUM_SLICES, IMAGE_SIZE, IMAGE_SIZE), dtype=torch.float32)

            # augmentation (train only)
            if self.aug and self.split == 'train' and len(dict_3d) > 0:
                rand_aug = get_rand_transform(mri_keys=list(dict_3d.keys()))
                dict_3d = rand_aug(dict_3d)

            for key in list(dict_3d.keys()):
                val = dict_3d[key]
                val = val.as_tensor() if hasattr(val, 'as_tensor') else val
                val = val.squeeze(0) if val.ndim == 4 else val
                dict_3d[key] = val

            for p in plane_order:                         # missing plane -> zeros
                if p not in dict_3d:
                    dict_3d[p] = torch.zeros((NUM_SLICES, IMAGE_SIZE, IMAGE_SIZE), dtype=torch.float32)

            if self.stack_neighbor:
                for key in plane_order:
                    dict_3d[key] = stack_neighbors(dict_3d[key], window=1)

        except Exception as e:
            print(f"[SKIP] Study {row.get('StudyInstanceUID')} failed: {e}")
            dict_3d = self._zeros_planes()

        list_3d = [dict_3d[p] for p in plane_order]
        meta = torch.from_numpy(np.stack([meta_dict[p] for p in plane_order]))

        if self.split == 'train':
            target = []
            for c in self.TARGETS:
                try:
                    target.append(float(row[c]))
                except Exception:
                    target.append(0.0)
            return list_3d, meta, torch.tensor(target, dtype=torch.float32)
        return list_3d, meta

# Model

## Self Supervised Learning (self distilation)

conv

In [10]:
# """
# In-domain SSL pretraining for the ConvNeXt-nano slice encoder — Knee MRI
# ===========================================================================

# Same self-distillation method as the DINOv2 SSL script (a student network
# learns to match a teacher that's a slow-moving average of the student — no
# labels needed). DINO's original paper tested this on a plain CNN backbone
# too, not just ViT, so it transfers directly to ConvNeXt.

# What's different from the DINOv2 version (everything else is identical):
# 1. Backbone is loaded with `timm`, not `transformers.AutoModel` — and the
#    checkpoint's old classifier head ("head.fc.*") is dropped before loading,
#    since num_classes=0 removes that layer from the model.
# 2. The backbone wrapper just returns `self.backbone(x)` directly — ConvNeXt
#    already pools to one vector per image, there's no CLS token to pull out.
# 3. The freeze function looks for blocks under `.stages[i].blocks[j]`
#    (ConvNeXt's layout), not `.encoder.layer` (the ViT layout).
# 4. Saved as a plain `state_dict` (.pth), not `save_pretrained()` — that
#    method belongs to `transformers` models, not timm ones.

# Everything else — multi-crop augmentation, the DINO loss, the two-LR-group
# optimizer, the EMA teacher update, and the linear-probe check on your 58
# ground-truth studies — is unchanged from the DINOv2 script.
# """
# # Assumed already imported earlier in the notebook in the DINOv2 script;
# # included here too so this file also runs standalone.
# import os
# import glob
# import random
# import warnings
# import numpy as np
# import torch
# import torch.nn as nn

# import math
# import pickle
# import torch.nn.functional as F
# import torchvision.transforms as T
# from PIL import Image
# from torch.utils.data import Dataset, DataLoader, RandomSampler

# import pydicom
# from pydicom.pixel_data_handlers.util import apply_voi_lut
# import timm

# from sklearn.linear_model import LogisticRegression
# from sklearn.model_selection import KFold
# from sklearn.metrics import roc_auc_score
# from sklearn.exceptions import UndefinedMetricWarning

# # ─────────────────────────────────────────────
# # Config — tune these for your GPU / time budget
# # ─────────────────────────────────────────────
# SERIES_ROOT    = "/kaggle/input/competitions/rsna-knee-abnormality-detection/train_series"
# BACKBONE_NAME  = "convnext_nano"
# BACKBONE_PATH  = "/kaggle/input/datasets/malihealavi/convnext-nano-pretrained-weights/convnext_nano_weights.pth"
# SSL_OUTPUT_DIR = "/kaggle/working/convnext_nano_knee_ssl"

# IMAGE_SIZE_GLOBAL = 256   # matches your main pipeline's IMAGE_SIZE
# IMAGE_SIZE_LOCAL  = 96
# NUM_GLOBAL_CROPS  = 2
# NUM_LOCAL_CROPS   = 4      # raise if you have GPU memory to spare — CNNs are cheaper than ViT here

# BATCH_SIZE      = 32
# ACCUM_STEPS     = 2        # effective batch size = BATCH_SIZE * ACCUM_STEPS
# EPOCHS          = 16        # start lower than the ViT run (20) — CNNs adapt faster; raise if the probe keeps improving
# STEPS_PER_EPOCH = 500        # caps how long one epoch takes; raise if you have time budget
# WARMUP_STEPS    = 2 * STEPS_PER_EPOCH

# # --- two learning rates instead of one ---------------------------
# HEAD_LR      = 1e-3 * (BATCH_SIZE * ACCUM_STEPS) / 256   # same formula/value as the DINOv2 script — used for the SSL head only
# BACKBONE_LR  = 1e-5                                         # much lower — only the unfrozen last blocks use this
# BASE_LR      = HEAD_LR
# # ---------------------------------------------------------------------------

# WEIGHT_DECAY = 0.04
# OUT_DIM      = 8192        # number of "prototypes" for the self-distillation loss
# STUDENT_TEMP = 0.1
# TEACHER_TEMP_WARMUP = 0.04
# TEACHER_TEMP = 0.07
# TEACHER_TEMP_WARMUP_EPOCHS = 3
# TEACHER_MOMENTUM_BASE = 0.996
# CENTER_MOMENTUM = 0.9

# SAVE_EVERY  = 1
# NUM_WORKERS = 4
# SEED        = 42

# # --- how much of the backbone stays trainable ---------------------
# NUM_TRAINABLE_BLOCKS = 2     # last 2 ConvNeXt blocks train; each block is much lighter than a ViT block, so you
#                              # can afford to raise this if the probe plateaus
# UNFREEZE_FINAL_NORM  = True  # also train norm_pre (cheap, sits right before pooling)
# # ---------------------------------------------------------------------------

# # --- linear probe settings -----------------------------------------
# RUN_LINEAR_PROBE   = True   # keep False until load_verified_probe_studies() below is filled in — script runs unchanged if False
# PROBE_EVERY        = 2      # epochs between probe checks
# PROBE_N_SLICES     = 5       # slices sampled per study for feature extraction (evenly spaced)
# PROBE_KFOLDS       = 5
# SSL_OUTPUT_DIR_BEST = SSL_OUTPUT_DIR + "_best_probe"  # separate folder — does NOT touch your existing SSL_OUTPUT_DIR checkpoints

# LABEL_COLUMNS = ['ACL', 'MCL', 'Medial Meniscus', 'Lateral Meniscus', 'Medial OA',
#                'Lateral OA', 'PF OA', 'Effusion', 'Synovitis', "Baker's", "Contusion", 'Fracture']
# # ---------------------------------------------------------------------------

# random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
# device = "cuda" if torch.cuda.is_available() else "cpu"
# torch.backends.cudnn.benchmark = True


# # ─────────────────────────────────────────────
# # 1. Build the list of usable series (reuse SERIES_FILES if you already built it)
# # ─────────────────────────────────────────────
# def build_series_index(root_dir):
#     series_dirs = glob.glob(os.path.join(root_dir, "*", "*"))
#     series_files = []
#     for sdir in series_dirs:
#         if not os.path.isdir(sdir):
#             continue
#         files = glob.glob(os.path.join(sdir, "*.dcm"))
#         if len(files) < 3:
#             continue
#         files.sort(key=lambda f: int(pydicom.dcmread(f, stop_before_pixels=True).InstanceNumber))
#         series_files.append(files)
#     return series_files

# with open('/kaggle/input/knee-abnormally-dataset/series_files.pkl', "rb") as f:
#     SERIES_FILES = pickle.load(f)
# print(len(SERIES_FILES))

# try:
#     ALL_SERIES_FILES = SERIES_FILES  # reuse if this notebook already built it
#     print(f"Reusing existing SERIES_FILES ({len(ALL_SERIES_FILES)} series)")
# except NameError:
#     ALL_SERIES_FILES = build_series_index(SERIES_ROOT)
#     print(f"Indexed {len(ALL_SERIES_FILES)} series")


# # ─────────────────────────────────────────────
# # 2. Slice-level dataset — every slice of every series, no labels
# # ─────────────────────────────────────────────
# def load_slice_as_array(path):
#     dicom = pydicom.dcmread(path)
#     try:
#         arr = apply_voi_lut(dicom.pixel_array, dicom)
#     except Exception:
#         arr = dicom.pixel_array
#     arr = arr.astype(np.float32)
#     lo, hi = np.percentile(arr, 1), np.percentile(arr, 99)
#     arr = np.clip((arr - lo) / (hi - lo + 1e-6), 0.0, 1.0)
#     return arr


# IMAGENET_MEAN = [0.485, 0.456, 0.406]
# IMAGENET_STD  = [0.229, 0.224, 0.225]


# class DINOMultiCrop:
#     """Student sees NUM_GLOBAL_CROPS large crops + NUM_LOCAL_CROPS small crops.
#     Teacher only ever sees the large crops (handled in the training loop)."""

#     def __init__(self):
#         flip_and_jitter = T.Compose([
#             T.RandomHorizontalFlip(p=0.5),
#             # brightness/contrast only — no hue/saturation, since the 3 channels
#             # here are neighboring slices, not real RGB color
#             T.RandomApply([T.ColorJitter(brightness=0.4, contrast=0.4)], p=0.6),
#             T.RandomApply([T.GaussianBlur(kernel_size=9, sigma=(0.1, 2.0))], p=0.4),
#         ])
#         normalize = T.Compose([T.ToTensor(), T.Normalize(IMAGENET_MEAN, IMAGENET_STD)])

#         self.global_tf = [T.Compose([
#             T.RandomResizedCrop(IMAGE_SIZE_GLOBAL, scale=(0.4, 1.0),
#                                  interpolation=T.InterpolationMode.BICUBIC),
#             flip_and_jitter, normalize,
#         ]) for _ in range(NUM_GLOBAL_CROPS)]

#         self.local_tf = T.Compose([
#             T.RandomResizedCrop(IMAGE_SIZE_LOCAL, scale=(0.05, 0.4),
#                                  interpolation=T.InterpolationMode.BICUBIC),
#             flip_and_jitter, normalize,
#         ])

#     def __call__(self, pil_img):
#         crops = [tf(pil_img) for tf in self.global_tf]
#         crops += [self.local_tf(pil_img) for _ in range(NUM_LOCAL_CROPS)]
#         return crops


# class SliceSSLDataset(Dataset):
#     """One item = one 2.5D slice (itself + its 2 neighbors as 3 channels),
#     pooled across ALL series/planes/sequence types."""

#     def __init__(self, series_files, transform):
#         self.series_files = series_files
#         self.transform = transform
#         self.index = [(si, zi) for si, files in enumerate(series_files) for zi in range(len(files))]

#     def __len__(self):
#         return len(self.index)

#     def __getitem__(self, idx):
#         si, zi = self.index[idx]
#         files = self.series_files[si]
#         n = len(files)
#         neighbor_idxs = [max(0, min(n - 1, zi + o)) for o in (-1, 0, 1)]
#         channels = [load_slice_as_array(files[i]) for i in neighbor_idxs]
#         img = np.stack(channels, axis=-1)  # (H, W, 3), float in [0,1]
#         pil_img = Image.fromarray((img * 255).astype(np.uint8), mode="RGB")
#         return self.transform(pil_img)


# def dino_collate(batch):
#     n_crops = len(batch[0])
#     return [torch.stack([sample[c] for sample in batch]) for c in range(n_crops)]


# dataset = SliceSSLDataset(ALL_SERIES_FILES, DINOMultiCrop())
# sampler = RandomSampler(dataset, replacement=True, num_samples=STEPS_PER_EPOCH * BATCH_SIZE)
# loader = DataLoader(dataset, batch_size=BATCH_SIZE, sampler=sampler,
#                      collate_fn=dino_collate, num_workers=NUM_WORKERS,
#                      pin_memory=True, drop_last=True)


# # ─────────────────────────────────────────────
# # 3. Student / teacher model
# # ─────────────────────────────────────────────
# class CNNBackboneWrapper(nn.Module):
#     """ConvNeXt already returns one pooled vector per image (no CLS token
#     to pull out like the ViT version) — just pass it through."""

#     def __init__(self, backbone):
#         super().__init__()
#         self.backbone = backbone

#     def forward(self, x):
#         return self.backbone(x)   # (B, num_features), already pooled


# class DINOHead(nn.Module):
#     def __init__(self, in_dim, out_dim=OUT_DIM, hidden_dim=2048, bottleneck_dim=256):
#         super().__init__()
#         self.mlp = nn.Sequential(
#             nn.Linear(in_dim, hidden_dim), nn.GELU(),
#             nn.Linear(hidden_dim, hidden_dim), nn.GELU(),
#             nn.Linear(hidden_dim, bottleneck_dim),
#         )
#         self.last_layer = nn.utils.weight_norm(nn.Linear(bottleneck_dim, out_dim, bias=False))
#         self.last_layer.weight_g.data.fill_(1)
#         self.last_layer.weight_g.requires_grad = False

#     def forward(self, x):
#         x = self.mlp(x)
#         x = F.normalize(x, dim=-1, p=2)
#         return self.last_layer(x)


# class DINOModel(nn.Module):
#     def __init__(self, backbone, embed_dim):
#         super().__init__()
#         self.backbone = CNNBackboneWrapper(backbone)
#         self.head = DINOHead(embed_dim)

#     def forward(self, x):
#         return self.head(self.backbone(x))


# # --- [FIX 1] freeze most of the backbone — ConvNeXt block layout -----------
# def freeze_convnext_except_last_n(backbone, n_trainable_blocks=NUM_TRAINABLE_BLOCKS,
#                                    unfreeze_final_norm=UNFREEZE_FINAL_NORM):
#     """timm ConvNeXt keeps its blocks under `.stages[i].blocks[j]`, not one
#     flat list like a ViT — flatten across stages first, then take the last N.
#     Run print(backbone) once if this doesn't match your timm version."""
#     for p in backbone.parameters():
#         p.requires_grad = False

#     all_blocks = [block for stage in backbone.stages for block in stage.blocks]
#     n_total = len(all_blocks)
#     n_trainable_blocks = min(n_trainable_blocks, n_total)
#     for block in all_blocks[n_total - n_trainable_blocks:]:
#         for p in block.parameters():
#             p.requires_grad = True

#     if unfreeze_final_norm and hasattr(backbone, "norm_pre"):
#         for p in backbone.norm_pre.parameters():
#             p.requires_grad = True

#     trainable = sum(p.numel() for p in backbone.parameters() if p.requires_grad)
#     total = sum(p.numel() for p in backbone.parameters())
#     print(f"[FIX 1] ConvNeXt backbone: {trainable:,} / {total:,} params trainable "
#           f"({100 * trainable / total:.1f}%) — last {n_trainable_blocks}/{n_total} blocks"
#           f"{' + norm_pre' if unfreeze_final_norm else ''} unfrozen")
# # ---------------------------------------------------------------------------


# # --- [FIX 2] ConvNeXt loading — timm, not transformers, and the old
# # classifier head ("head.fc.*") must be dropped since num_classes=0 removes
# # that layer from the model ---------------------------------------------
# def load_convnext(path):
#     m = timm.create_model(BACKBONE_NAME, pretrained=False, num_classes=0)
#     sd = torch.load(path, map_location="cpu")
#     sd = {k: v for k, v in sd.items() if not k.startswith("head.fc")}
#     missing, unexpected = m.load_state_dict(sd, strict=False)
#     assert not unexpected, f"Unexpected keys — architecture mismatch: {unexpected}"
#     return m
# # ---------------------------------------------------------------------------


# student_backbone = load_convnext(BACKBONE_PATH)
# teacher_backbone = load_convnext(BACKBONE_PATH)
# embed_dim = student_backbone.num_features

# freeze_convnext_except_last_n(student_backbone)

# student = DINOModel(student_backbone, embed_dim).to(device)
# teacher = DINOModel(teacher_backbone, embed_dim).to(device)
# teacher.load_state_dict(student.state_dict())  # sync the randomly-init heads too
# for p in teacher.parameters():
#     p.requires_grad = False


# def unwrap(m):
#     """DataParallel stores the real model under `.module` — use this
#     whenever you need to reach a custom attribute (e.g. .backbone)."""
#     return m.module if isinstance(m, nn.DataParallel) else m


# if torch.cuda.device_count() > 1:
#     print(f"Using {torch.cuda.device_count()} GPUs via DataParallel")
#     student = nn.DataParallel(student)
#     teacher = nn.DataParallel(teacher)


# # ─────────────────────────────────────────────
# # 4. DINO loss (self-distillation cross-entropy + centering, no labels)
# # ─────────────────────────────────────────────
# class DINOLoss(nn.Module):
#     def __init__(self, out_dim):
#         super().__init__()
#         self.register_buffer("center", torch.zeros(1, out_dim))
#         self.teacher_temp_schedule = np.concatenate((
#             np.linspace(TEACHER_TEMP_WARMUP, TEACHER_TEMP, TEACHER_TEMP_WARMUP_EPOCHS),
#             np.full(max(EPOCHS - TEACHER_TEMP_WARMUP_EPOCHS, 0), TEACHER_TEMP),
#         ))

#     def forward(self, student_outputs, teacher_outputs, epoch):
#         temp = self.teacher_temp_schedule[min(epoch, len(self.teacher_temp_schedule) - 1)]
#         teacher_probs = [F.softmax((t - self.center) / temp, dim=-1).detach() for t in teacher_outputs]
#         student_logp  = [F.log_softmax(s / STUDENT_TEMP, dim=-1) for s in student_outputs]

#         loss, n_terms = 0.0, 0
#         for ti, tp in enumerate(teacher_probs):
#             for si, slp in enumerate(student_logp):
#                 if si == ti:  # never compare a view against itself
#                     continue
#                 loss += torch.sum(-tp * slp, dim=-1).mean()
#                 n_terms += 1
#         loss = loss / n_terms

#         with torch.no_grad():
#             batch_center = torch.cat(teacher_outputs).mean(dim=0, keepdim=True)
#             self.center = self.center * CENTER_MOMENTUM + batch_center * (1 - CENTER_MOMENTUM)
#         return loss


# dino_loss = DINOLoss(OUT_DIM).to(device)

# # --- two parameter groups, two learning rates -----------------------
# # Only params with requires_grad=True need gradients/updates — after
# # freeze_convnext_except_last_n(), that's just the last few blocks (+ maybe
# # norm_pre) in the backbone, plus the whole head.
# backbone_trainable_params = [p for p in unwrap(student).backbone.parameters() if p.requires_grad]
# head_params = list(unwrap(student).head.parameters())

# BACKBONE_GROUP_IDX, HEAD_GROUP_IDX = 0, 1
# optimizer = torch.optim.AdamW(
#     [
#         {"params": backbone_trainable_params, "lr": BACKBONE_LR},
#         {"params": head_params, "lr": HEAD_LR},
#     ],
#     weight_decay=WEIGHT_DECAY,
# )
# # ---------------------------------------------------------------------------

# scaler = torch.cuda.amp.GradScaler()


# def cosine_lr(step, total_steps, base_lr, warmup_steps):
#     if step < warmup_steps:
#         return base_lr * step / max(1, warmup_steps)
#     progress = (step - warmup_steps) / max(1, total_steps - warmup_steps)
#     return 0.5 * base_lr * (1 + math.cos(math.pi * progress))


# def cosine_momentum(epoch, total_epochs, base=TEACHER_MOMENTUM_BASE):
#     return 1 - (1 - base) * (math.cos(math.pi * epoch / total_epochs) + 1) / 2


# @torch.no_grad()
# def update_teacher(m):
#     # Still correct with a partially frozen backbone: frozen student params
#     # never change, and a frozen pair (pt == ps already) stays exactly equal
#     # under this EMA formula (pt*m + ps*(1-m) == ps when pt == ps). Only the
#     # unfrozen blocks actually drift and get EMA'd normally.
#     for ps, pt in zip(student.parameters(), teacher.parameters()):
#         pt.data.mul_(m).add_(ps.data, alpha=1 - m)


# FREEZE_LAST_LAYER_EPOCHS = 1  # standard DINO stabilization trick — without this,
#                               # a randomly-initialized output layer can drag the
#                               # whole network into a collapsed shortcut early on


# def cancel_last_layer_grad(model, epoch):
#     """Zero the output layer's gradient for the first epoch. This is what
#     the official DINO code calls --freeze_last_layer (default 1 epoch)."""
#     if epoch >= FREEZE_LAST_LAYER_EPOCHS:
#         return
#     for n, p in unwrap(model).head.named_parameters():
#         if "last_layer" in n and p.grad is not None:
#             p.grad = None


# # ─────────────────────────────────────────────
# # 4b. Linear probe — the ground-truth check for feature quality
# #     (loss going down does NOT mean the features are good; this does)
# # ─────────────────────────────────────────────
# import ast
# def load_verified_probe_studies():
#     global TRUE_LABELS_DF, LIGHT_DF , SERIES_MERGED
#     final_list = []
#     for _, row in TRUE_LABELS_DF.iterrows():
#         study_id = row['StudyInstanceUID']
#         study_df = LIGHT_DF[LIGHT_DF['StudyInstanceUID'] == study_id]
#         if study_df.empty:
#             print(f"  [probe] no files found for {study_id}, skipping")
#             continue
#         label_vector = row[LABEL_COLUMNS].values.astype(float)
#         for _, series_row in study_df.iterrows():
#             final_list.append((ast.literal_eval(series_row['file_path']), label_vector, study_id))
#     return final_list


# @torch.no_grad()
# def extract_study_embedding(teacher_model, files, n_slices=PROBE_N_SLICES):
#     """Mean-pools teacher embeddings over a few evenly spaced 2.5D
#     slices of one series — same slice-building logic as SliceSSLDataset,
#     just without augmentation (resize only, no crop/flip/jitter)."""
#     n = len(files)
#     if n == 0:
#         return None
#     idxs = np.linspace(0, n - 1, num=min(n_slices, n), dtype=int)
#     normalize = T.Compose([
#         T.Resize((IMAGE_SIZE_GLOBAL, IMAGE_SIZE_GLOBAL), interpolation=T.InterpolationMode.BICUBIC),
#         T.ToTensor(), T.Normalize(IMAGENET_MEAN, IMAGENET_STD),
#     ])
#     embeds = []
#     for zi in idxs:
#         zi = int(zi)
#         neighbor_idxs = [max(0, min(n - 1, zi + o)) for o in (-1, 0, 1)]
#         channels = [load_slice_as_array(files[i]) for i in neighbor_idxs]
#         img = np.stack(channels, axis=-1)
#         pil_img = Image.fromarray((img * 255).astype(np.uint8), mode="RGB")
#         x = normalize(pil_img).unsqueeze(0).to(device)
#         feat = teacher_model.backbone(x)  # CNNBackboneWrapper -> pooled vector
#         embeds.append(feat.squeeze(0).float().cpu().numpy())
#     return np.mean(embeds, axis=0)

# from sklearn.model_selection import GroupKFold

# def run_linear_probe(teacher_model, verified_studies, epoch):
#     """
#     Extracts one pooled embedding per verified study, then fits a simple
#     per-label logistic regression with k-fold cross-validation (58 samples
#     is too few to trust a single train/val split) and reports mean AUC
#     across labels and folds. This is the real signal for whether SSL
#     pretraining is helping — not the training loss.
#     """
#     teacher_model.eval()
#     feats, labels , groups = [], [], []
#     for files, label_vec , study_id in verified_studies:
#         emb = extract_study_embedding(teacher_model, files)
#         if emb is None:
#             continue
#         feats.append(emb)
#         labels.append(label_vec)
#         groups.append(study_id)
        
#     feats = np.stack(feats)
#     labels = np.array(labels)

#     n_labels = labels.shape[1]
#     n_groups = len(set(groups))
#     n_splits = max(2, min(PROBE_KFOLDS, n_groups))
#     kf = GroupKFold(n_splits=n_splits)
#     per_label_aucs = []
 
#     with warnings.catch_warnings():
#         warnings.simplefilter("ignore", category=UndefinedMetricWarning)
#         for label_idx in range(n_labels):
#             y = labels[:, label_idx]
#             if len(np.unique(y)) < 2:
#                 continue  # AUC undefined if a label has only one class overall
#             fold_aucs = []
#             for train_idx, val_idx in kf.split(feats, groups=groups):
#                 y_train = (y[train_idx] > 0.5).astype(int)
#                 if len(np.unique(y_train)) < 2:
#                     continue
#                 y_val = (y[val_idx] > 0.5).astype(int)
#                 if len(np.unique(y_val)) < 2:
#                     continue
#                 clf = LogisticRegression(max_iter=1000)
#                 clf.fit(feats[train_idx], y_train)
#                 probs = clf.predict_proba(feats[val_idx])[:, 1]
#                 fold_aucs.append(roc_auc_score(y_val, probs))
#             if fold_aucs:
#                 per_label_aucs.append(np.mean(fold_aucs))
 
#     mean_auc = float(np.mean(per_label_aucs)) if per_label_aucs else float("nan")
#     print(f"  [linear probe] epoch {epoch + 1}: mean AUC over "
#           f"{len(per_label_aucs)}/{n_labels} usable labels = {mean_auc:.4f}")
#     return mean_auc
 



# verified_probe_studies = None
# if RUN_LINEAR_PROBE:
#     verified_probe_studies = load_verified_probe_studies()
#     print(f"[FIX 3] Loaded {len(verified_probe_studies)} verified studies "
#           f"for linear-probe checks (every {PROBE_EVERY} epochs)")
# class _ProbeWrapper(nn.Module):
#     def __init__(self, backbone_wrapper):
#         super().__init__()
#         self.backbone = backbone_wrapper

# baseline_container = _ProbeWrapper(
#     CNNBackboneWrapper(load_convnext(BACKBONE_PATH))
# ).to(device)

# baseline_auc = run_linear_probe(baseline_container, verified_probe_studies, epoch=-1)
# print(f"Baseline (plain ConvNeXt-nano, no SSL) probe AUC: {baseline_auc:.4f}")

# best_probe_auc = -1.0
# probe_history = []


# # --- [FIX 4] save/load as a plain state_dict, not save_pretrained ----------
# # save_pretrained() is a `transformers` method — it doesn't exist on a timm
# # model. Save the backbone's state_dict directly, the same way you already
# # load your other convnext checkpoint.
# SSL_WEIGHTS_FILENAME = "convnext_nano_ssl.pth"
# def save_convnext(wrapped_model, out_dir):
#     os.makedirs(out_dir, exist_ok=True)
#     path = os.path.join(out_dir, SSL_WEIGHTS_FILENAME)
#     torch.save(unwrap(wrapped_model).backbone.backbone.state_dict(), path)
#     return path
# # ---------------------------------------------------------------------------


# # ─────────────────────────────────────────────
# # 5. Training loop
# # ─────────────────────────────────────────────
# os.makedirs(SSL_OUTPUT_DIR, exist_ok=True)
# total_steps = STEPS_PER_EPOCH * EPOCHS
# global_step = 0
# loss_history = []

# for epoch in range(EPOCHS):
#     student.train()
#     teacher.eval()
#     running = 0.0

#     for it, crops in enumerate(loader):
#         crops = [c.to(device, non_blocking=True) for c in crops]

#         # one cosine schedule, two peak LRs -----------------
#         # cosine_lr(..., base_lr=1.0, ...) returns the 0..1 warmup/decay
#         # multiplier by itself; multiply it by each group's own peak LR.
#         mult = cosine_lr(global_step, total_steps, 1.0, WARMUP_STEPS)
#         lr_backbone = BACKBONE_LR * mult
#         lr_head = HEAD_LR * mult
#         optimizer.param_groups[BACKBONE_GROUP_IDX]["lr"] = lr_backbone
#         optimizer.param_groups[HEAD_GROUP_IDX]["lr"] = lr_head
#         # ------------------------------------------------------------------

#         with torch.cuda.amp.autocast():
#             student_outputs = [student(c) for c in crops]
#             with torch.no_grad():
#                 teacher_outputs = [teacher(c) for c in crops[:NUM_GLOBAL_CROPS]]
#             loss = dino_loss(student_outputs, teacher_outputs, epoch) / ACCUM_STEPS

#         scaler.scale(loss).backward()

#         if (it + 1) % ACCUM_STEPS == 0:
#             scaler.unscale_(optimizer)
#             cancel_last_layer_grad(student, epoch)
#             torch.nn.utils.clip_grad_norm_(student.parameters(), max_norm=3.0)
#             scaler.step(optimizer)
#             scaler.update()
#             optimizer.zero_grad()
#             update_teacher(cosine_momentum(epoch, EPOCHS))

#         running += loss.item() * ACCUM_STEPS
#         global_step += 1

#     epoch_loss = running / STEPS_PER_EPOCH
#     loss_history.append(epoch_loss)
#     print(f"epoch {epoch + 1:>3}/{EPOCHS}  loss={epoch_loss:.4f}  "
#           f"lr_backbone={lr_backbone:.2e}  lr_head={lr_head:.2e}")

#     if (epoch + 1) % SAVE_EVERY == 0 or epoch == EPOCHS - 1:
#         path = save_convnext(teacher, SSL_OUTPUT_DIR)
#         print(f"  saved teacher backbone -> {path}")

#     # --- periodic linear probe check ----------------------------------
#     if RUN_LINEAR_PROBE and ((epoch + 1) % PROBE_EVERY == 0 or epoch == EPOCHS - 1):
#         probe_auc = run_linear_probe(unwrap(teacher), verified_probe_studies, epoch)
#         probe_history.append((epoch + 1, probe_auc))
#         if probe_auc == probe_auc and probe_auc > best_probe_auc:  # NaN-safe check
#             best_probe_auc = probe_auc
#             path = save_convnext(teacher, SSL_OUTPUT_DIR_BEST)
#             print(f"  new best probe AUC ({best_probe_auc:.4f}) -> saved to {path}")
#     # ------------------------------------------------------------------

# print("Done. Load the result later in your SliceEncoder like this:")
# print(f'  m = timm.create_model("{BACKBONE_NAME}", pretrained=False, num_classes=0)')
# print(f'  m.load_state_dict(torch.load(os.path.join("{SSL_OUTPUT_DIR}", "{SSL_WEIGHTS_FILENAME}"), map_location="cpu"))')
# if RUN_LINEAR_PROBE:
#     print(f"Best probe AUC = {best_probe_auc:.4f}, saved separately at {SSL_OUTPUT_DIR_BEST}")
#     print(f"Probe history: {probe_history}")


dino

In [11]:
# """
# In-domain SSL pretraining for the DINOv2 slice encoder — Knee MRI
# ===================================================================

# Why this method (short version):
# - The SliceEncoder backbone is DINOv2, which was itself pretrained with
#   "self-distillation" (a student network learns to match a teacher
#   network's output; the teacher is a slow-moving average of the student —
#   no labels needed at all). Continuing pretraining with the SAME technique
#   on your own knee MRI slices is the most natural way to shift the model's
#   features from "natural photos" to "knee MRI" before you ever touch labels.
# - Alternatives like MAE (mask patches, reconstruct pixels) or SimCLR/MoCo
#   (contrastive, needs big batches or a memory bank) either need much more
#   data/epochs to pay off, or fit a ViT-DINOv2 starting point less naturally.
# - This script uses a LIGHTER version of the full DINOv2 recipe (it skips
#   the extra patch-level loss and regularizer DINOv2 adds on top of DINOv1)
#   since those mainly help at ImageNet scale and cost a lot of extra
#   compute for a Kaggle GPU budget, without much payoff at this data size.

# What it does:
# 1. Builds a slice-level dataset from every knee MRI series (all planes /
#    sequence types pooled together — no labels used or needed; the same
#    SliceEncoder will see all of them later, so SSL should too).
# 2. For each slice, builds a 3-channel image from the slice + its 2 spatial
#    neighbors (matches how your SliceEncoder later sees data).
# 3. Runs "multi-crop" self-distillation: a student sees 2 large crops + a
#    few small crops of the same slice, a teacher (EMA = running average of
#    the student) only sees the large crops, and the student is trained to
#    predict the teacher's output for every crop it doesn't share a view with.
# 4. Saves the teacher's backbone in HuggingFace format so you can load it
#    exactly the way you already do:
#        AutoModel.from_pretrained(SSL_OUTPUT_DIR, local_files_only=True)

# Tuning knobs are all in the CONFIG block below — see the notes at the
# bottom of the chat message for what to change and why.

# """

# import pickle
# import torch.nn.functional as F
# import torchvision.transforms as T
# from PIL import Image
# from torch.utils.data import Dataset, DataLoader, RandomSampler

# import pydicom
# from pydicom.pixel_data_handlers.util import apply_voi_lut
# from transformers import AutoModel

# # [FIX 3] needed only for the linear probe — sklearn ships by default on Kaggle
# from sklearn.linear_model import LogisticRegression
# from sklearn.model_selection import KFold
# from sklearn.metrics import roc_auc_score
# from sklearn.exceptions import UndefinedMetricWarning

# # ─────────────────────────────────────────────
# # Config — tune these for your GPU / time budget
# # ─────────────────────────────────────────────
# SERIES_ROOT    = "/kaggle/input/competitions/rsna-knee-abnormality-detection/train_series"
# BACKBONE_PATH  = "/kaggle/input/models/metaresearch/dinov2/pytorch/small/1"  # your existing DINOv2-small checkpoint
# SSL_OUTPUT_DIR = "/kaggle/working/dinov2_knee_ssl"  # matches the path your SliceEncoder already expects

# IMAGE_SIZE_GLOBAL = 256   # matches your main pipeline's IMAGE_SIZE
# IMAGE_SIZE_LOCAL  = 96
# NUM_GLOBAL_CROPS  = 2
# NUM_LOCAL_CROPS   = 4      # raise to 6-8 if you have GPU memory to spare

# BATCH_SIZE      = 32
# ACCUM_STEPS     = 2        # effective batch size = BATCH_SIZE * ACCUM_STEPS
# EPOCHS          = 20
# STEPS_PER_EPOCH = 500      # caps how long one epoch takes; raise if you have time budget
# WARMUP_STEPS    = 2 * STEPS_PER_EPOCH

# # --- [FIX 2] two learning rates instead of one ---------------------------
# HEAD_LR      = 1e-3 * (BATCH_SIZE * ACCUM_STEPS) / 256   # same formula/value as your original BASE_LR — used for the SSL head only
# BACKBONE_LR  = 1e-5                                        # NEW, much lower — only the unfrozen last blocks use this
# BASE_LR      = HEAD_LR   # kept so anything else referencing BASE_LR still works, unchanged meaning for the head
# # ---------------------------------------------------------------------------

# WEIGHT_DECAY = 0.04
# OUT_DIM      = 8192        # number of "prototypes" for the self-distillation loss
#                             # (smaller than the usual 65536 since our dataset is far
#                             # smaller than ImageNet)
# STUDENT_TEMP = 0.1
# TEACHER_TEMP_WARMUP = 0.04
# TEACHER_TEMP = 0.07
# TEACHER_TEMP_WARMUP_EPOCHS = 5
# TEACHER_MOMENTUM_BASE = 0.996
# CENTER_MOMENTUM = 0.9

# SAVE_EVERY  = 4
# NUM_WORKERS = 4
# SEED        = 42

# # --- [FIX 1] how much of the backbone stays trainable ---------------------
# NUM_TRAINABLE_BLOCKS = 2     # only the LAST 2 transformer blocks train; set to 1 for an even more conservative fine-tune
# UNFREEZE_FINAL_NORM  = True  # also train the small final layernorm right after the blocks (cheap, sits right on the CLS output)
# # ---------------------------------------------------------------------------

# # --- [FIX 3] linear probe settings -----------------------------------------
# RUN_LINEAR_PROBE   = True   # keep False until load_verified_probe_studies() below is filled in — script runs unchanged if False
# PROBE_EVERY        = 4      # epochs between probe checks
# PROBE_N_SLICES     = 5       # slices sampled per study for feature extraction (evenly spaced)
# PROBE_KFOLDS       = 5
# SSL_OUTPUT_DIR_BEST = SSL_OUTPUT_DIR + "_best_probe"  # separate folder — does NOT touch your existing SSL_OUTPUT_DIR checkpoints

# LABEL_COLUMNS = ['ACL', 'MCL', 'Medial Meniscus', 'Lateral Meniscus', 'Medial OA',
#                'Lateral OA', 'PF OA', 'Effusion', 'Synovitis', "Baker's", "Contusion", 'Fracture']
# # ---------------------------------------------------------------------------

# random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
# device = "cuda" if torch.cuda.is_available() else "cpu"
# torch.backends.cudnn.benchmark = True


# # ─────────────────────────────────────────────
# # 1. Build the list of usable series (reuse SERIES_FILES if you already built it)
# # ─────────────────────────────────────────────
# def build_series_index(root_dir):
#     series_dirs = glob.glob(os.path.join(root_dir, "*", "*"))
#     series_files = []
#     for sdir in series_dirs:
#         if not os.path.isdir(sdir):
#             continue
#         files = glob.glob(os.path.join(sdir, "*.dcm"))
#         if len(files) < 3:
#             continue
#         files.sort(key=lambda f: int(pydicom.dcmread(f, stop_before_pixels=True).InstanceNumber))
#         series_files.append(files)
#     return series_files

# with open('/kaggle/input/datasets/malihealavi/knee-abnormally-dataset/series_files.pkl', "rb") as f:
#     SERIES_FILES = pickle.load(f)
# print(len(SERIES_FILES))

# try:
#     ALL_SERIES_FILES = SERIES_FILES  # reuse if this notebook already built it
#     print(f"Reusing existing SERIES_FILES ({len(ALL_SERIES_FILES)} series)")
# except NameError:
#     ALL_SERIES_FILES = build_series_index(SERIES_ROOT)
#     print(f"Indexed {len(ALL_SERIES_FILES)} series")


# # ─────────────────────────────────────────────
# # 2. Slice-level dataset — every slice of every series, no labels
# # ─────────────────────────────────────────────
# def load_slice_as_array(path):
#     dicom = pydicom.dcmread(path)
#     try:
#         arr = apply_voi_lut(dicom.pixel_array, dicom)
#     except Exception:
#         arr = dicom.pixel_array
#     arr = arr.astype(np.float32)
#     lo, hi = np.percentile(arr, 1), np.percentile(arr, 99)
#     arr = np.clip((arr - lo) / (hi - lo + 1e-6), 0.0, 1.0)
#     return arr


# IMAGENET_MEAN = [0.485, 0.456, 0.406]
# IMAGENET_STD  = [0.229, 0.224, 0.225]


# class DINOMultiCrop:
#     """Student sees NUM_GLOBAL_CROPS large crops + NUM_LOCAL_CROPS small crops.
#     Teacher only ever sees the large crops (handled in the training loop)."""

#     def __init__(self):
#         flip_and_jitter = T.Compose([
#             T.RandomHorizontalFlip(p=0.5),
#             # brightness/contrast only — no hue/saturation, since the 3 channels
#             # here are neighboring slices, not real RGB color
#             T.RandomApply([T.ColorJitter(brightness=0.4, contrast=0.4)], p=0.6),
#             T.RandomApply([T.GaussianBlur(kernel_size=9, sigma=(0.1, 2.0))], p=0.4),
#         ])
#         normalize = T.Compose([T.ToTensor(), T.Normalize(IMAGENET_MEAN, IMAGENET_STD)])

#         self.global_tf = [T.Compose([
#             T.RandomResizedCrop(IMAGE_SIZE_GLOBAL, scale=(0.4, 1.0),
#                                  interpolation=T.InterpolationMode.BICUBIC),
#             flip_and_jitter, normalize,
#         ]) for _ in range(NUM_GLOBAL_CROPS)]

#         self.local_tf = T.Compose([
#             T.RandomResizedCrop(IMAGE_SIZE_LOCAL, scale=(0.05, 0.4),
#                                  interpolation=T.InterpolationMode.BICUBIC),
#             flip_and_jitter, normalize,
#         ])

#     def __call__(self, pil_img):
#         crops = [tf(pil_img) for tf in self.global_tf]
#         crops += [self.local_tf(pil_img) for _ in range(NUM_LOCAL_CROPS)]
#         return crops


# class SliceSSLDataset(Dataset):
#     """One item = one 2.5D slice (itself + its 2 neighbors as 3 channels),
#     pooled across ALL series/planes/sequence types."""

#     def __init__(self, series_files, transform):
#         self.series_files = series_files
#         self.transform = transform
#         self.index = [(si, zi) for si, files in enumerate(series_files) for zi in range(len(files))]

#     def __len__(self):
#         return len(self.index)

#     def __getitem__(self, idx):
#         si, zi = self.index[idx]
#         files = self.series_files[si]
#         n = len(files)
#         neighbor_idxs = [max(0, min(n - 1, zi + o)) for o in (-1, 0, 1)]
#         channels = [load_slice_as_array(files[i]) for i in neighbor_idxs]
#         img = np.stack(channels, axis=-1)  # (H, W, 3), float in [0,1]
#         pil_img = Image.fromarray((img * 255).astype(np.uint8), mode="RGB")
#         return self.transform(pil_img)


# def dino_collate(batch):
#     n_crops = len(batch[0])
#     return [torch.stack([sample[c] for sample in batch]) for c in range(n_crops)]


# dataset = SliceSSLDataset(ALL_SERIES_FILES, DINOMultiCrop())
# sampler = RandomSampler(dataset, replacement=True, num_samples=STEPS_PER_EPOCH * BATCH_SIZE)
# loader = DataLoader(dataset, batch_size=BATCH_SIZE, sampler=sampler,
#                      collate_fn=dino_collate, num_workers=NUM_WORKERS,
#                      pin_memory=True, drop_last=True)


# # ─────────────────────────────────────────────
# # 3. Student / teacher model
# # ─────────────────────────────────────────────
# class HFBackboneWrapper(nn.Module):
#     """Wraps the HF DINOv2 model to return just the CLS embedding —
#     identical to what your SliceEncoder does."""

#     def __init__(self, backbone):
#         super().__init__()
#         self.backbone = backbone

#     def forward(self, x):
#         return self.backbone(x).last_hidden_state[:, 0, :]


# class DINOHead(nn.Module):
#     def __init__(self, in_dim, out_dim=OUT_DIM, hidden_dim=2048, bottleneck_dim=256):
#         super().__init__()
#         self.mlp = nn.Sequential(
#             nn.Linear(in_dim, hidden_dim), nn.GELU(),
#             nn.Linear(hidden_dim, hidden_dim), nn.GELU(),
#             nn.Linear(hidden_dim, bottleneck_dim),
#         )
#         self.last_layer = nn.utils.weight_norm(nn.Linear(bottleneck_dim, out_dim, bias=False))
#         self.last_layer.weight_g.data.fill_(1)
#         self.last_layer.weight_g.requires_grad = False

#     def forward(self, x):
#         x = self.mlp(x)
#         x = F.normalize(x, dim=-1, p=2)
#         return self.last_layer(x)


# class DINOModel(nn.Module):
#     def __init__(self, backbone, embed_dim):
#         super().__init__()
#         self.backbone = HFBackboneWrapper(backbone)
#         self.head = DINOHead(embed_dim)

#     def forward(self, x):
#         return self.head(self.backbone(x))


# # --- [FIX 1] freeze most of the backbone -----------------------------------
# def freeze_backbone_except_last_n(backbone, n_trainable_blocks=NUM_TRAINABLE_BLOCKS,
#                                    unfreeze_final_norm=UNFREEZE_FINAL_NORM):
#     """
#     Freezes all backbone parameters (requires_grad=False, so no gradient
#     is computed and the optimizer never touches them), then unfreezes only
#     the LAST `n_trainable_blocks` transformer blocks (and, optionally, the
#     final layernorm right after them).

#     Expects the standard HuggingFace Dinov2Model layout:
#         backbone.embeddings            <- patch/pos embeddings, stays frozen
#         backbone.encoder.layer[i]      <- transformer blocks
#         backbone.layernorm             <- final norm, optionally trainable

#     If your checkpoint has a different internal structure, this will raise
#     a clear error rather than silently freezing the wrong thing.
#     """
#     for p in backbone.parameters():
#         p.requires_grad = False

#     try:
#         blocks = backbone.encoder.layer
#     except AttributeError as e:
#         raise AttributeError(
#             "Could not find backbone.encoder.layer — this backbone's "
#             "internal structure is different than expected. Run "
#             "print(backbone) to inspect it and update this function to "
#             "point at the actual transformer-block list before continuing."
#         ) from e

#     n_total = len(blocks)
#     n_trainable_blocks = min(n_trainable_blocks, n_total)
#     for block in blocks[n_total - n_trainable_blocks:]:
#         for p in block.parameters():
#             p.requires_grad = True

#     if unfreeze_final_norm and hasattr(backbone, "layernorm"):
#         for p in backbone.layernorm.parameters():
#             p.requires_grad = True

#     trainable = sum(p.numel() for p in backbone.parameters() if p.requires_grad)
#     total = sum(p.numel() for p in backbone.parameters())
#     print(f"[FIX 1] Backbone: {trainable:,} / {total:,} params trainable "
#           f"({100 * trainable / total:.1f}%) — last {n_trainable_blocks}/{n_total} blocks"
#           f"{' + final layernorm' if unfreeze_final_norm else ''} unfrozen")
# # ---------------------------------------------------------------------------


# student_backbone = AutoModel.from_pretrained(BACKBONE_PATH, local_files_only=True)
# teacher_backbone = AutoModel.from_pretrained(BACKBONE_PATH, local_files_only=True)
# embed_dim = student_backbone.config.hidden_size

# # [FIX 1] apply the freeze to the student's backbone before it's wrapped.
# # (Not needed on teacher_backbone — a few lines below, ALL teacher params
# # are set to requires_grad=False anyway, since the teacher is never
# # trained by gradient descent, only by the EMA update further down.)
# freeze_backbone_except_last_n(student_backbone)

# student = DINOModel(student_backbone, embed_dim).to(device)
# teacher = DINOModel(teacher_backbone, embed_dim).to(device)
# teacher.load_state_dict(student.state_dict())  # sync the randomly-init heads too
# for p in teacher.parameters():
#     p.requires_grad = False


# def unwrap(m):
#     """DataParallel stores the real model under `.module` — use this
#     whenever you need to reach a custom attribute (e.g. .backbone)."""
#     return m.module if isinstance(m, nn.DataParallel) else m


# if torch.cuda.device_count() > 1:
#     print(f"Using {torch.cuda.device_count()} GPUs via DataParallel")
#     student = nn.DataParallel(student)
#     teacher = nn.DataParallel(teacher)


# # ─────────────────────────────────────────────
# # 4. DINO loss (self-distillation cross-entropy + centering, no labels)
# # ─────────────────────────────────────────────
# class DINOLoss(nn.Module):
#     def __init__(self, out_dim):
#         super().__init__()
#         self.register_buffer("center", torch.zeros(1, out_dim))
#         self.teacher_temp_schedule = np.concatenate((
#             np.linspace(TEACHER_TEMP_WARMUP, TEACHER_TEMP, TEACHER_TEMP_WARMUP_EPOCHS),
#             np.full(max(EPOCHS - TEACHER_TEMP_WARMUP_EPOCHS, 0), TEACHER_TEMP),
#         ))

#     def forward(self, student_outputs, teacher_outputs, epoch):
#         temp = self.teacher_temp_schedule[min(epoch, len(self.teacher_temp_schedule) - 1)]
#         teacher_probs = [F.softmax((t - self.center) / temp, dim=-1).detach() for t in teacher_outputs]
#         student_logp  = [F.log_softmax(s / STUDENT_TEMP, dim=-1) for s in student_outputs]

#         loss, n_terms = 0.0, 0
#         for ti, tp in enumerate(teacher_probs):
#             for si, slp in enumerate(student_logp):
#                 if si == ti:  # never compare a view against itself
#                     continue
#                 loss += torch.sum(-tp * slp, dim=-1).mean()
#                 n_terms += 1
#         loss = loss / n_terms

#         with torch.no_grad():
#             batch_center = torch.cat(teacher_outputs).mean(dim=0, keepdim=True)
#             self.center = self.center * CENTER_MOMENTUM + batch_center * (1 - CENTER_MOMENTUM)
#         return loss


# dino_loss = DINOLoss(OUT_DIM).to(device)

# # --- [FIX 2] two parameter groups, two learning rates -----------------------
# # Only params with requires_grad=True need gradients/updates — after
# # freeze_backbone_except_last_n(), that's just the last few blocks (+ maybe
# # the final layernorm) in the backbone, plus the whole head.
# backbone_trainable_params = [p for p in unwrap(student).backbone.parameters() if p.requires_grad]
# head_params = list(unwrap(student).head.parameters())

# BACKBONE_GROUP_IDX, HEAD_GROUP_IDX = 0, 1
# optimizer = torch.optim.AdamW(
#     [
#         {"params": backbone_trainable_params, "lr": BACKBONE_LR},
#         {"params": head_params, "lr": HEAD_LR},
#     ],
#     weight_decay=WEIGHT_DECAY,
# )
# # ---------------------------------------------------------------------------

# scaler = torch.cuda.amp.GradScaler()


# def cosine_lr(step, total_steps, base_lr, warmup_steps):
#     if step < warmup_steps:
#         return base_lr * step / max(1, warmup_steps)
#     progress = (step - warmup_steps) / max(1, total_steps - warmup_steps)
#     return 0.5 * base_lr * (1 + math.cos(math.pi * progress))


# def cosine_momentum(epoch, total_epochs, base=TEACHER_MOMENTUM_BASE):
#     return 1 - (1 - base) * (math.cos(math.pi * epoch / total_epochs) + 1) / 2


# @torch.no_grad()
# def update_teacher(m):
#     # Still correct with a partially frozen backbone: frozen student params
#     # never change, and a frozen pair (pt == ps already) stays exactly equal
#     # under this EMA formula (pt*m + ps*(1-m) == ps when pt == ps). Only the
#     # unfrozen blocks actually drift and get EMA'd normally.
#     for ps, pt in zip(student.parameters(), teacher.parameters()):
#         pt.data.mul_(m).add_(ps.data, alpha=1 - m)


# FREEZE_LAST_LAYER_EPOCHS = 1  # standard DINO stabilization trick — without this,
#                               # a randomly-initialized output layer can drag the
#                               # whole network into a collapsed shortcut early on


# def cancel_last_layer_grad(model, epoch):
#     """Zero the output layer's gradient for the first epoch. This is what
#     the official DINO code calls --freeze_last_layer (default 1 epoch)."""
#     if epoch >= FREEZE_LAST_LAYER_EPOCHS:
#         return
#     for n, p in unwrap(model).head.named_parameters():
#         if "last_layer" in n and p.grad is not None:
#             p.grad = None


# # ─────────────────────────────────────────────
# # 4b. [FIX 3] Linear probe — the ground-truth check for feature quality
# #     (loss going down does NOT mean the features are good; this does)
# # ─────────────────────────────────────────────
# import ast
# def load_verified_probe_studies():
#     global TRUE_LABELS_DF, LIGHT_DF
#     final_list = []
#     for _, row in TRUE_LABELS_DF.iterrows():
#         study_id = row['StudyInstanceUID']
#         study_df = LIGHT_DF[LIGHT_DF['StudyInstanceUID'] == study_id]
#         if study_df.empty:
#             print(f"  [probe] no files found for {study_id}, skipping")
#             continue
#         label_vector = row[LABEL_COLUMNS].values.astype(float)
#         for _, series_row in study_df.iterrows():
#             final_list.append((ast.literal_eval(series_row['file_path']), label_vector, study_id))
#     return final_list


# @torch.no_grad()
# def extract_study_embedding(teacher_model, files, n_slices=PROBE_N_SLICES):
#     """Mean-pools teacher CLS embeddings over a few evenly spaced 2.5D
#     slices of one series — same slice-building logic as SliceSSLDataset,
#     just without augmentation (resize only, no crop/flip/jitter)."""
#     n = len(files)
#     if n == 0:
#         return None
#     idxs = np.linspace(0, n - 1, num=min(n_slices, n), dtype=int)
#     normalize = T.Compose([
#         T.Resize((IMAGE_SIZE_GLOBAL, IMAGE_SIZE_GLOBAL), interpolation=T.InterpolationMode.BICUBIC),
#         T.ToTensor(), T.Normalize(IMAGENET_MEAN, IMAGENET_STD),
#     ])
#     embeds = []
#     for zi in idxs:
#         zi = int(zi)
#         neighbor_idxs = [max(0, min(n - 1, zi + o)) for o in (-1, 0, 1)]
#         channels = [load_slice_as_array(files[i]) for i in neighbor_idxs]
#         img = np.stack(channels, axis=-1)
#         pil_img = Image.fromarray((img * 255).astype(np.uint8), mode="RGB")
#         x = normalize(pil_img).unsqueeze(0).to(device)
#         feat = teacher_model.backbone(x)  # HFBackboneWrapper -> CLS embedding
#         embeds.append(feat.squeeze(0).float().cpu().numpy())
#     return np.mean(embeds, axis=0)

# from sklearn.model_selection import GroupKFold

# def run_linear_probe(teacher_model, verified_studies, epoch):
#     """
#     Extracts one pooled embedding per verified study, then fits a simple
#     per-label logistic regression with k-fold cross-validation (58 samples
#     is too few to trust a single train/val split) and reports mean AUC
#     across labels and folds. This is the real signal for whether SSL
#     pretraining is helping — not the training loss.
#     """
#     teacher_model.eval()
#     feats, labels , groups = [], [], []
#     for files, label_vec , study_id in verified_studies:
#         emb = extract_study_embedding(teacher_model, files)
#         if emb is None:
#             continue
#         feats.append(emb)
#         labels.append(label_vec)
#         groups.append(study_id)
        
#     feats = np.stack(feats)
#     labels = np.array(labels)

#     n_labels = labels.shape[1]
#     n_groups = len(set(groups))
#     n_splits = max(2, min(PROBE_KFOLDS, n_groups))
#     kf = GroupKFold(n_splits=n_splits)
#     per_label_aucs = []
 
#     with warnings.catch_warnings():
#         warnings.simplefilter("ignore", category=UndefinedMetricWarning)
#         for label_idx in range(n_labels):
#             y = labels[:, label_idx]
#             if len(np.unique(y)) < 2:
#                 continue  # AUC undefined if a label has only one class overall
#             fold_aucs = []
#             for train_idx, val_idx in kf.split(feats, groups=groups):
#                 y_train = (y[train_idx] > 0.5).astype(int)
#                 if len(np.unique(y_train)) < 2:
#                     continue
#                 y_val = (y[val_idx] > 0.5).astype(int)
#                 if len(np.unique(y_val)) < 2:
#                     continue
#                 clf = LogisticRegression(max_iter=1000)
#                 clf.fit(feats[train_idx], y_train)
#                 probs = clf.predict_proba(feats[val_idx])[:, 1]
#                 fold_aucs.append(roc_auc_score(y_val, probs))
#             if fold_aucs:
#                 per_label_aucs.append(np.mean(fold_aucs))
 
#     mean_auc = float(np.mean(per_label_aucs)) if per_label_aucs else float("nan")
#     print(f"  [linear probe] epoch {epoch + 1}: mean AUC over "
#           f"{len(per_label_aucs)}/{n_labels} usable labels = {mean_auc:.4f}")
#     return mean_auc
 



# # verified_probe_studies = None
# # if RUN_LINEAR_PROBE:
# #     verified_probe_studies = load_verified_probe_studies()
# #     print(f"[FIX 3] Loaded {len(verified_probe_studies)} verified studies "
# #           f"for linear-probe checks (every {PROBE_EVERY} epochs)")
# # class _ProbeWrapper(nn.Module):
# #     def __init__(self, backbone_wrapper):
# #         super().__init__()
# #         self.backbone = backbone_wrapper

# # baseline_container = _ProbeWrapper(
# #     HFBackboneWrapper(AutoModel.from_pretrained(BACKBONE_PATH, local_files_only=True))
# # ).to(device)

# # baseline_auc = run_linear_probe(baseline_container, verified_probe_studies, epoch=-1)
# # print(f"Baseline (plain DINOv2, no SSL) probe AUC: {baseline_auc:.4f}")

# # best_probe_auc = -1.0
# # probe_history = []


# # # ─────────────────────────────────────────────
# # # 5. Training loop
# # # ─────────────────────────────────────────────
# # os.makedirs(SSL_OUTPUT_DIR, exist_ok=True)
# # total_steps = STEPS_PER_EPOCH * EPOCHS
# # global_step = 0
# # loss_history = []

# # for epoch in range(EPOCHS):
# #     student.train()
# #     teacher.eval()
# #     running = 0.0

# #     for it, crops in enumerate(loader):
# #         crops = [c.to(device, non_blocking=True) for c in crops]

# #         # --- [FIX 2] one cosine schedule, two peak LRs -----------------
# #         # cosine_lr(..., base_lr=1.0, ...) returns the 0..1 warmup/decay
# #         # multiplier by itself; multiply it by each group's own peak LR.
# #         mult = cosine_lr(global_step, total_steps, 1.0, WARMUP_STEPS)
# #         lr_backbone = BACKBONE_LR * mult
# #         lr_head = HEAD_LR * mult
# #         optimizer.param_groups[BACKBONE_GROUP_IDX]["lr"] = lr_backbone
# #         optimizer.param_groups[HEAD_GROUP_IDX]["lr"] = lr_head
# #         # ------------------------------------------------------------------

# #         with torch.cuda.amp.autocast():
# #             student_outputs = [student(c) for c in crops]
# #             with torch.no_grad():
# #                 teacher_outputs = [teacher(c) for c in crops[:NUM_GLOBAL_CROPS]]
# #             loss = dino_loss(student_outputs, teacher_outputs, epoch) / ACCUM_STEPS

# #         scaler.scale(loss).backward()

# #         if (it + 1) % ACCUM_STEPS == 0:
# #             scaler.unscale_(optimizer)
# #             cancel_last_layer_grad(student, epoch)
# #             torch.nn.utils.clip_grad_norm_(student.parameters(), max_norm=3.0)
# #             scaler.step(optimizer)
# #             scaler.update()
# #             optimizer.zero_grad()
# #             update_teacher(cosine_momentum(epoch, EPOCHS))

# #         running += loss.item() * ACCUM_STEPS
# #         global_step += 1

# #     epoch_loss = running / STEPS_PER_EPOCH
# #     loss_history.append(epoch_loss)
# #     print(f"epoch {epoch + 1:>3}/{EPOCHS}  loss={epoch_loss:.4f}  "
# #           f"lr_backbone={lr_backbone:.2e}  lr_head={lr_head:.2e}")

# #     if (epoch + 1) % SAVE_EVERY == 0 or epoch == EPOCHS - 1:
# #         unwrap(teacher).backbone.backbone.save_pretrained(SSL_OUTPUT_DIR)
# #         print(f"  saved teacher backbone -> {SSL_OUTPUT_DIR}")

# #     # --- [FIX 3] periodic linear probe check ---------------------------
# #     if RUN_LINEAR_PROBE and ((epoch + 1) % PROBE_EVERY == 0 or epoch == EPOCHS - 1):
# #         probe_auc = run_linear_probe(unwrap(teacher), verified_probe_studies, epoch)
# #         probe_history.append((epoch + 1, probe_auc))
# #         if probe_auc == probe_auc and probe_auc > best_probe_auc:  # NaN-safe check
# #             best_probe_auc = probe_auc
# #             os.makedirs(SSL_OUTPUT_DIR_BEST, exist_ok=True)
# #             unwrap(teacher).backbone.backbone.save_pretrained(SSL_OUTPUT_DIR_BEST)
# #             print(f"  new best probe AUC ({best_probe_auc:.4f}) -> saved to {SSL_OUTPUT_DIR_BEST}")
# #     # ------------------------------------------------------------------

# # print("Done. Load the result the same way you already do:")
# # print(f'  AutoModel.from_pretrained("{SSL_OUTPUT_DIR}", local_files_only=True)')
# # if RUN_LINEAR_PROBE:
# #     print(f"Best probe AUC = {best_probe_auc:.4f}, saved separately at {SSL_OUTPUT_DIR_BEST}")
# #     print(f"Probe history: {probe_history}")

## Standard Supervised Learning

In [12]:

# ─────────────────────────────────────────────
# Shared 2D CNN slice encoder
# ─────────────────────────────────────────────
class SliceEncoder(nn.Module):
    """
    Input:  (B*N_m, 3, H, W)   — flat batch of triple-stacked slices
    Output: (B*N_m, d)          — one feature vector per slice
    """
    def __init__(self, d: int = 256, bb: str = None):
        super().__init__()
        bb = bb if bb is not None else ENCODER_BB   # read at call time
        self.bb = bb
        self.proj = None

        if bb == 'convnext_nano':
            backbone = timm.create_model("convnext_nano", pretrained=False, num_classes=0)
            loaded = torch.load(
                '/kaggle/input/datasets/malihealavi/knee-pretrained-encoder/ssl_convnext_nano.pth',
                #/kaggle/input/datasets/malihealavi/knee-pretrained-encoder/ssl_convnext_nano.pth
                map_location="cpu",
                weights_only=False,
            )
            sd = loaded.state_dict() if hasattr(loaded, "state_dict") else loaded
            sd = {k: v for k, v in sd.items() if not k.startswith("head.fc")}
            missing, unexpected = backbone.load_state_dict(sd, strict=False)
            assert not unexpected, f"Unexpected keys — architecture mismatch: {unexpected}"
            print(f'Selected Encoder: {bb}')
            # --- Count and print parameters ---
            total_params = sum(p.numel() for p in backbone.parameters())
            trainable_params = sum(p.numel() for p in backbone.parameters() if p.requires_grad)
            print(f"Total Parameters: {total_params:,}")
            print(f"Trainable Parameters: {trainable_params:,}")
            # ----------------------------------
            feature = backbone.num_features        # e.g. 640 for convnext_nano
            self.proj = nn.Linear(feature, d)
            for p in backbone.parameters():         # freeze backbone
                p.requires_grad = False
            trainable_params = sum(p.numel() for p in backbone.parameters() if p.requires_grad)
            print(f"Trainable Parameters: {trainable_params:,}")
        elif bb == 'dinov2_small':
            DINOV2_LOCAL_PATH = "/kaggle/input/datasets/malihealavi/knee-pretrained-encoder/ssl_dinov2_small"
            # DINOV2_LOCAL_PATH = "/kaggle/input/rsna-knee-model/dinov2_knee_ssl_best_probe"
            backbone = AutoModel.from_pretrained(DINOV2_LOCAL_PATH, local_files_only=True)
            feature = backbone.config.hidden_size          # 768 for dinov2-base
            self.proj = nn.Linear(feature, d)                # dino has no head to swap, so project manually
            print(f'Selected Encoder: {bb}')
            for p in backbone.parameters():
                p.requires_grad = False

        elif bb == 'dinov3_small':
            DINOV2_LOCAL_PATH = "/kaggle/input/models/metaresearch/dinov2/pytorch/small/1"
            backbone = AutoModel.from_pretrained(DINOV2_LOCAL_PATH, local_files_only=True)
            feature = backbone.config.hidden_size          # 768 for dinov2-base
            self.proj = nn.Linear(feature, d)                # dino has no head to swap, so project manually
            print(f'Selected Encoder: {bb}')
            for p in backbone.parameters():
                p.requires_grad = False

        else:
            raise ValueError(f"Unknown backbone: {bb}")

        self.backbone = backbone

    def forward(self, x):
        if self.bb == 'dinov2_small':
            outputs = self.backbone(x)
            x = outputs.last_hidden_state[:, 0, :]   # CLS token, (B*N_m, hidden)
            x = self.proj(x)                          # -> (B*N_m, d)
        else:
            x = self.backbone(x)                       # resnet: (B*N_m, d) already
            if self.proj is not None:                   # convnext: (B*N_m, feature) -> proj -> (B*N_m, d)
                x = self.proj(x)
        return x 
    
# ─────────────────────────────────────────────
#      Within-modality slice attention
# ─────────────────────────────────────────────
class SliceAttentionPooler(nn.Module):
    """
    Input:  (B, N_m, d)  + optional mask (B, N_m)
    Output: (B, d)        — one vector per modality per case
    """
    def __init__(self, d: int = 256, nhead: int = 4,
                 num_layers: int = 2, max_slices: int = NUM_SLICES):
        super().__init__()
        self.pos_emb   = nn.Embedding(max_slices, d)
        # cls_token : a vector with size d that concat to slices of each modality ==>  shape changes from (B, N, d) to (B, N + 1, d)
        self.cls_token = nn.Parameter(torch.randn(1, 1, d)) 
        encoder_layer  = nn.TransformerEncoderLayer(
            d_model=d, nhead=nhead, dim_feedforward=d * 4,
            dropout=0.1, batch_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)

    def forward(self, x, key_padding_mask=None):
        B, N, d = x.shape
        # add positional encoding
        pos = torch.arange(N, device=x.device) 
        x = x + self.pos_emb(pos)
        # Copies the CLS token for every item in the batch (1 cls per modality)
        cls = self.cls_token.expand(B, -1, -1)
        x = torch.cat([cls, x], dim=1)

        if key_padding_mask is not None:
            cls_mask = torch.zeros(B, 1, dtype=torch.bool, device=x.device)
            key_padding_mask = torch.cat([cls_mask, key_padding_mask], dim=1)
            
        x = self.transformer(x, src_key_padding_mask=key_padding_mask) 
        return x[:, 0] 

# ─────────────────────────────────────────────
#  Cross-modality fusion attention
# ─────────────────────────────────────────────
class CrossModalFusion(nn.Module):
    """
    Input:  list of M tensors each (B, d) | for each Modality we have B vectors with size d
    Output: (B, d)  — fused across all modalities | all Modalities merge together so One vector per case
    """
    def __init__(self, d: int = 256, nhead: int = 4,
                 num_layers: int = 2, num_modalities: int = 3):
        super().__init__()
        self.mod_emb = nn.Embedding(num_modalities, d)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d, nhead=nhead, dim_feedforward=d * 4,
            dropout=0.1, batch_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer,
                                                  num_layers=num_layers)

    def forward(self, modality_vecs):
        # modality_vecs: list of 4 tensors each (B, d)
        x   = torch.stack(modality_vecs, dim=1)         # (B, 4, d)
        ids = torch.arange(x.size(1), device=x.device)
        x   = x + self.mod_emb(ids)                     # (B, 4, d)
        x   = self.transformer(x)                        # (B, 4, d)
        return x.mean(dim=1)                             # (B, d)



class MRI25DFusionModelViewRoutingDualEncoder(nn.Module):
    def __init__(self,
                 d: int = 256,
                 nhead: int = 4,
                 num_layers: int = 2,
                 num_modalities: int = 3,
                 max_slices: int = NUM_SLICES,
                 num_metadata_features : int= 33,
                 d_meta: int = 32):
        super().__init__()
        self.d = d 
        self.dino_slice_encoder = SliceEncoder(d=d, bb='dinov2_small')
        self.conv_slice_encoder = SliceEncoder(d=d, bb='convnext_nano')
        self.slice_fuse = nn.Sequential(
                    nn.Linear(d * 2, d),
                    nn.LayerNorm(d),
                )
        self.slice_poolers = nn.ModuleList([
            SliceAttentionPooler(d=d, nhead=nhead,
                                 num_layers=num_layers,
                                 max_slices=max_slices)
            for _ in range(num_modalities)
        ])

        self.fusion = CrossModalFusion(d=d, nhead=nhead,
                                       num_layers=num_layers,
                                       num_modalities=num_modalities)

        # ── Target-specific view routing ────────────────────────────
        # Order below MUST match your true label column order.
        # Check against LIGHT_DF and fix if different.
        self.LABEL_ORDER = [
            "acl", "mcl", "medial_meniscus", "lateral_meniscus",
            "medial_oa", "lateral_oa", "pf_oa", "effusion", "synovitis",
            "bakers_cyst", "contusion", "fracture",
        ]
        AXIAL_LABELS   = ["pf_oa", "effusion", "synovitis"]
        SC_LABELS      = ["acl", "mcl", "medial_meniscus", "lateral_meniscus", "medial_oa", "lateral_oa"]
        GENERAL_LABELS = ["bakers_cyst", "contusion", "fracture"]
        grouped = AXIAL_LABELS + SC_LABELS + GENERAL_LABELS
        perm = [grouped.index(l) for l in self.LABEL_ORDER]
        self.register_buffer("label_perm", torch.tensor(perm, dtype=torch.long))

        def make_head(out_dim):
            return nn.Sequential(
                nn.Linear(d*2, d ),
                nn.ReLU(),
                nn.Dropout(0.2),
                nn.Linear(d , out_dim),
            )

        self.axial_head   = make_head(len(AXIAL_LABELS))    # PF OA, Effusion, Synovitis
        self.sc_head      = make_head(len(SC_LABELS))       # ACL, MCL, Menisci, Medial/Lateral OA
        self.general_head = make_head(len(GENERAL_LABELS))  # Baker's cyst, Contusion, Fracture
        # self.metadata_encoder = MetadataEncoder(num_features= num_metadata_features, d=d_meta)
        # self.meta_fuse = nn.Linear(d + d_meta, d)

    def forward(self, modalities, meta=None, masks=None):
        if masks is None:
            masks = [None] * len(modalities)
 
        modality_vecs = []
        for slices, pooler, mask in zip(modalities, self.slice_poolers, masks):
            B, N_m, C, H, W = slices.shape
            flat = slices.reshape(B * N_m, C, H, W) 
            feats_dino = self.dino_slice_encoder(flat).view(B, N_m, -1)
            feats_conv = self.conv_slice_encoder(flat).view(B, N_m, -1)
            assert feats_dino.shape[-1] == self.d and feats_conv.shape[-1] == self.d
            
            feats_cat = self.slice_fuse(torch.cat([feats_dino, feats_conv], dim=-1))
            pooled = pooler(feats_cat, mask)
            modality_vecs.append(pooled)
 
        fused   = self.fusion(modality_vecs)
        axial   = modality_vecs[0]
        sc      = (modality_vecs[1] + modality_vecs[2]) / 2
        all_avg = torch.stack(modality_vecs, dim=1).mean(dim=1)      # (B, d)

        out = torch.cat([self.axial_head(torch.cat([axial, fused], dim=1)),
                         self.sc_head(torch.cat([sc, fused], dim=1)),
                         self.general_head(torch.cat([all_avg, fused], dim=1)),
                        ], dim=1)
        return out[:, self.label_perm]



# Training

In [13]:
# ──────────────────────────────────────────────────────────────────────────────
# Plotting helper
# ──────────────────────────────────────────────────────────────────────────────
def plot_loss(val_losses, train_losses, val_auc, path: str) -> None:
    epochs = range(1, len(train_losses) + 1)
    plt.figure(figsize=(8, 4))
    plt.plot(epochs, train_losses, label="Train Loss",  color="blue",  marker="o", linewidth=2)
    plt.plot(epochs, val_losses,   label="Val Loss",    color="red",   marker="o", linewidth=2)
    plt.plot(epochs, val_auc,      label="Val AUC",     color="green", marker="o", linewidth=2)
    # plt.title(f"Fold {fold} – Loss & AUC over Epochs", fontsize=14)
    plt.xlabel("Epoch")
    plt.ylabel("Value")
    plt.xticks(epochs)
    plt.legend()
    plt.grid(True, linestyle="--", alpha=0.6)
    plt.tight_layout()
    plt.savefig(path, dpi=150, bbox_inches="tight")
    plt.close()
    print(f"  Plot saved → {os.path.abspath(path)}")

# ── HELPER FUNCTION ───────────────
# ── HELPER FUNCTION ───────────────
import psutil, os, ctypes

def free_ram():
    gc.collect()
    torch.cuda.empty_cache()
    ctypes.CDLL("libc.so.6").malloc_trim(0)   # gives freed memory back to the OS
import threading, time, psutil, os, shutil

def _cgroup_mem():
    for cur, lim in [("/sys/fs/cgroup/memory.current", "/sys/fs/cgroup/memory.max"),
                     ("/sys/fs/cgroup/memory/memory.usage_in_bytes", "/sys/fs/cgroup/memory/memory.limit_in_bytes")]:
        try:
            return int(open(cur).read()), open(lim).read().strip()
        except Exception:
            pass
    return None, None

def start_mem_watcher(path="/kaggle/working/mem_log.txt", every=30):
    def loop():
        main = psutil.Process(os.getpid())
        while True:
            try:
                kids = main.children(recursive=True)
                kids_rss = sum(k.memory_info().rss for k in kids)
                cur, lim = _cgroup_mem()
                shm = shutil.disk_usage("/dev/shm").used
                gpu = " ".join(f"gpu{i}={torch.cuda.memory_reserved(i)/1e9:.1f}GB"
                               for i in range(torch.cuda.device_count()))
                with open(path, "a") as f:
                    f.write(f"{time.strftime('%H:%M:%S')} main={main.memory_info().rss/1e9:.1f}GB "
                            f"workers({len(kids)})={kids_rss/1e9:.1f}GB shm={shm/1e9:.1f}GB "
                            f"cgroup={(cur or 0)/1e9:.1f}GB limit={lim} {gpu}\n")
            except Exception:
                pass
            time.sleep(every)
    threading.Thread(target=loop, daemon=True).start()

def blend(img_p, meta_p, w, eps=1e-6):
    """final = image logit + w * metadata logit   (w = 0 -> image only)."""
    z = logit(np.clip(img_p, eps, 1 - eps)) + w * logit(np.clip(meta_p, eps, 1 - eps))
    return expit(z)
 
def macro_auc(y, p):
    a = []
    for j in range(y.shape[1]):
        try:
            # Check if column has both classes before calculating
            if len(np.unique(y[:, j])) < 2:
                continue
            score = roc_auc_score(y[:, j], p[:, j])
            if not np.isnan(score):
                a.append(score)
        except ValueError:
            pass
            
    # Return 0.5 (random guessing) or 0.0 if no valid columns could be evaluated
    if len(a) == 0:
        return float('nan') # or 0.5 depending on your preference
    return float(np.mean(a))

def search_blend_weight(img_val, meta_val, y_val, img_gt, meta_gt, y_gt,
                        weights=None, n_boot=200, seed=0, verbose=True):
    """
    Choose ONE global weight w.
      1. AUC of the blend on val for every w (val = the big split, 441 studies).
      2. Bootstrap the val studies -> standard error (SE) + 95% range of the gain over w=0.
      3. One-SE rule: take the SMALLEST w whose val AUC is within 1 SE of the best val AUC.
      4. Use the blend only if: gain 95% range is above 0 on val AND GT AUC does not drop.
    """
    if weights is None:
        weights = np.arange(0, 1.01, 0.1)
    weights = sorted({round(float(w), 2) for w in weights} | {0.0})
 
    rng = np.random.default_rng(seed)
    N = len(y_val)
    boots = rng.integers(0, N, size=(n_boot, N))
 
    val_auc, gt_auc, boot_auc = {}, {}, {}
    for w in weights:
        p_val = blend(img_val, meta_val, w)
        val_auc[w] = macro_auc(y_val, p_val)
        gt_auc[w] = macro_auc(y_gt, blend(img_gt, meta_gt, w))
        boot_auc[w] = np.array([macro_auc(y_val[b], p_val[b]) for b in boots])
    gains = {w: boot_auc[w] - boot_auc[0.0] for w in weights}
 
    best = max(weights, key=lambda w: val_auc[w])
    se = float(boot_auc[best].std())
    chosen = min(w for w in weights if val_auc[w] >= val_auc[best] - se)
    lo, hi = np.percentile(gains[chosen], [2.5, 97.5])
    gt_ok = gt_auc[chosen] >= gt_auc[0.0]
    use_blend = (chosen > 0) and (lo > 0) and gt_ok
    final_w = chosen if use_blend else 0.0
 
    if verbose:
        print(f"{'w':>5} | {'val AUC':>8} | {'val gain vs w=0 (95% range)':>30} | {'GT AUC':>7}")
        for w in weights:
            l, h = np.percentile(gains[w], [2.5, 97.5])
            mark = "  <- chosen" if w == chosen else ("  <- best on val" if w == best else "")
            print(f"{w:>5} | {val_auc[w]:8.4f} | {gains[w].mean():+8.4f} ({l:+.4f} to {h:+.4f}) | "
                  f"{gt_auc[w]:7.4f}{mark}")
        print(f"\nbest w on val = {best} | bootstrap SE = {se:.4f} | chosen w (1-SE rule) = {chosen}")
        print(f"gain at chosen w: 95% range {lo:+.4f} to {hi:+.4f} | "
              f"GT AUC {gt_auc[0.0]:.4f} -> {gt_auc[chosen]:.4f}")
        print("DECISION:", f"use blend with w = {final_w}" if use_blend
              else "do NOT blend (use the image model alone, w = 0)")
    return dict(final_w=final_w, chosen_w=chosen, best_w=best, se=se, use_blend=use_blend,
                val_auc=val_auc, gt_auc=gt_auc)
 
def unwrap(m):
    """DataParallel stores the real model under `.module` — use this
    whenever you need to reach a custom attribute (e.g. .backbone)."""
    return m.module if isinstance(m, nn.DataParallel) else m

def _z(p, eps=1e-6):
    return logit(np.clip(p, eps, 1 - eps))
 
 
def _auc1(y, s):
    try:
        return roc_auc_score(y, s)
    except ValueError:                           # only one class
        return np.nan
 
 
def _fit_stack(z_img, z_meta, y, C=1.0, nonneg=True):
    """Logistic regression  P(y=1) = sigmoid(a*z_img + b*z_meta + c)  with an L2 penalty on a, b.
    Starts from a=1, b=0 (= image only). Returns array [a, b, c]."""
    if y.min() == y.max():                       # one class only: nothing to learn
        return np.array([1.0, 0.0, 0.0])
    X = np.column_stack([z_img, z_meta])
    n = len(y)
    lam = 1.0 / (C * n)
 
    def f(theta):
        s = X @ theta[:2] + theta[2]
        p = expit(s)
        loss = np.mean(np.logaddexp(0.0, s) - y * s) + 0.5 * lam * (theta[0] ** 2 + theta[1] ** 2)
        r = p - y
        grad = np.array([np.mean(r * X[:, 0]) + lam * theta[0],
                         np.mean(r * X[:, 1]) + lam * theta[1],
                         np.mean(r)])
        return loss, grad
 
    bounds = [(0, None), (0, None), (None, None)] if nonneg else [(None, None)] * 3
    res = minimize(f, x0=np.array([1.0, 0.0, 0.0]), jac=True, method="L-BFGS-B", bounds=bounds)
    return res.x
 
 
def fit_label_weights(img_val, meta_val, y_val, img_gt, meta_gt, y_gt,
                      C=1.0, nonneg=True, n_splits=5, seed=0):
    """
    One (a, b, c) per label, fitted on VAL only.
      val AUC with metadata = honest: weights are fitted on 4/5 of val and scored on the other 1/5.
      GT  AUC with metadata = honest: GT is never used for fitting (weights come from all of val).
    All inputs are (N, 12): predicted probabilities (img_*, meta_*) and 0/1 labels (y_*).
    """
    L = y_val.shape[1]
    zi_v, zm_v, zi_g, zm_g = _z(img_val), _z(meta_val), _z(img_gt), _z(meta_gt)
 
    coefs = np.zeros((L, 3))
    oof = np.zeros_like(zi_v)                                   # held-out val scores
    kf = KFold(n_splits, shuffle=True, random_state=seed)
    for j in range(L):
        y = y_val[:, j]
        coefs[j] = _fit_stack(zi_v[:, j], zm_v[:, j], y, C, nonneg)       # final weights
        for tr, te in kf.split(zi_v):
            c = _fit_stack(zi_v[tr, j], zm_v[tr, j], y[tr], C, nonneg)
            oof[te, j] = expit(c[0] * zi_v[te, j] + c[1] * zm_v[te, j] + c[2])
 
    a, b = coefs[:, 0], coefs[:, 1]
    w = np.where(a > 1e-6, b / np.maximum(a, 1e-6), np.inf)
    out = dict(coefs=coefs, w=w,
               val_img=np.array([_auc1(y_val[:, j], zi_v[:, j]) for j in range(L)]),
               val_blend=np.array([_auc1(y_val[:, j], oof[:, j]) for j in range(L)]),
               gt_img=np.array([_auc1(y_gt[:, j], zi_g[:, j]) for j in range(L)]),
               gt_blend=np.array([_auc1(y_gt[:, j], a[j] * zi_g[:, j] + b[j] * zm_g[:, j])
                                  for j in range(L)]))
    return out
 
 
def print_label_weights(res, epoch=None, names=LABELS):
    t = f"Epoch {epoch} | " if epoch is not None else ""
    print(f"\n{t}per-label weights  (img = image only, +meta = with metadata; val is held-out)")
    print(f"{'label':17s} | {'w=b/a':>6} | {'val img':>7} {'+meta':>7} | {'GT img':>7} {'+meta':>7}")
    for j, n in enumerate(names):
        w = "meta" if np.isinf(res["w"][j]) else f"{res['w'][j]:.2f}"
        print(f"{n:17s} | {w:>6} | {res['val_img'][j]:7.3f} {res['val_blend'][j]:7.3f} | "
              f"{res['gt_img'][j]:7.3f} {res['gt_blend'][j]:7.3f}")
    m = {k: np.nanmean(res[k]) for k in ["val_img", "val_blend", "gt_img", "gt_blend"]}
    print(f"{'MACRO':17s} | {'':>6} | {m['val_img']:7.3f} {m['val_blend']:7.3f} | "
          f"{m['gt_img']:7.3f} {m['gt_blend']:7.3f}")
    print(f"gain with metadata: val {m['val_blend'] - m['val_img']:+.4f} | GT {m['gt_blend'] - m['gt_img']:+.4f}")
 
 
def apply_label_weights(img_p, meta_p, coefs):
    """Final probabilities (N, 12) with the fitted per-label weights (use for the test set)."""
    return expit(coefs[:, 0] * _z(img_p) + coefs[:, 1] * _z(meta_p) + coefs[:, 2])

# ── collate for TRAIN, VAL  ──────────────────────────────────────

def mri_collate_fn(batch):
    num_modalities = len(batch[0][0])
    modalities_out = [
        torch.stack([item[0][m] for item in batch], dim=0)
        for m in range(num_modalities)
    ]
    meta_out = torch.stack([item[1] for item in batch])
    labels_out = torch.stack([item[2] for item in batch])
    return modalities_out, meta_out, labels_out

# ── collate for test (no labels) ──────────────────────────────────────
def mri_collate_fn_test(batch):
    num_modalities = len(batch[0][0])                      # item[0] is the list of 3 planes
    modalities_out = [
        torch.stack([item[0][m] for item in batch], dim=0)  # item[0][m], not item[m]
        for m in range(num_modalities)
    ]
    meta_out = torch.stack([item[1] for item in batch])
    return modalities_out, meta_out

import gc, os, ctypes, psutil

_libc = ctypes.CDLL("libc.so.6")
_proc = psutil.Process(os.getpid())

def cleanup(tag=""):
    gc.collect()                  # 1st: free Python objects
    torch.cuda.empty_cache()      # 2nd: give unused GPU cache back
    try:
        _libc.malloc_trim(0)      # 3rd: give free CPU memory back to the system
    except Exception:
        pass
    gpu = ", ".join(f"{torch.cuda.memory_reserved(i)/1e9:.1f}"
                    for i in range(torch.cuda.device_count()))
    print(f"[cleanup {tag}] RAM {_proc.memory_info().rss/1e9:.2f} GB | GPU reserved {gpu} GB")

In [14]:
class TrainerSingleFold():
    def __init__(self,
                 train_df: pd.DataFrame,
                 epochs: int,
                 device: str,
                 scaler: Optional[torch.amp.GradScaler] = None):  # FIX WARN 1

        tmp_ds = KneeDataset(train_labeled=train_df.head(1), series_merged=LIGHT_DF,
                     split='train', augmentation=False)
        num_meta = tmp_ds.num_meta
        print("num metadata features:", num_meta)
        
        self.model = MRI25DFusionModelViewRoutingDualEncoder(
            d=256, nhead=4, num_layers=2,
            num_modalities=3, max_slices=NUM_SLICES,
            num_metadata_features=num_meta, d_meta=32
        )

        
        self.model.to(device)
        # if torch.cuda.device_count() > 1:
        #     print(f"Using {torch.cuda.device_count()} GPUs")
        #     self.model = nn.DataParallel(self.model)
        
        self.epochs    = epochs

        self.backbone_params = [p for n, p in self.model.named_parameters()
                             if "slice_encoder.backbone" in n]
        self.head_params = [p for n, p in self.model.named_parameters()
                             if "slice_encoder.backbone" not in n]
        
        self.optimizer = optim.AdamW([
            {"params": self.backbone_params, "lr": 1e-5},
            {"params": self.head_params,     "lr": 1e-4},
        ], weight_decay=5e-3)
        
        self.device    = device
        self.scaler    = scaler
        self.accumulation_steps = ACCUMULATION_STEP
        
        self.train, val        = train_test_split(train_df, test_size=0.1, random_state=42)
        self.val   = val.reset_index(drop=True)
        self.GT_VAL   = GT_VAL.reset_index(drop=True) #58 Ground Truth samples
        

        self.val_ds   = KneeDataset(train_labeled = self.val , series_merged = LIGHT_DF, split='train', augmentation=False)
        self.gt_val_ds   = KneeDataset(train_labeled = self.GT_VAL , series_merged = LIGHT_DF, split='train', augmentation=False)

        self.criterion    = nn.BCEWithLogitsLoss()
        self.best_auc     = 0.0
        
        self.scheduler = CosineAnnealingLR(self.optimizer, T_max=NUM_CYCLE, eta_min=5e-7)
        self.current_cycle =None
        self.current_epoch = None
        self.train_loss =[]
        self.valid_loss =[]
        self.valid_auc=[]
        self.patience = 0
        self.stall = 0
        self.resume_cycle = 0
        self.resume_epoch = -1
        self.epoch_preds = {}
        self.blend_log = []
        ckpt_result = self.load_checkpoint()   # keep this call, but capture the return value
        if ckpt_result is not None:
            self.resume_cycle, self.resume_epoch = ckpt_result

    def _unfreeze_slice_encoder(self, cycle):
        global BATCH_SIZE, ACCUMULATION_STEP
    
        if cycle < UNFREEZE_START_CYCLE:
            n_unfrozen = 0
        else:
            n_unfrozen = 1 + (cycle - UNFREEZE_START_CYCLE) // CYCLES_PER_BLOCK
    
        n_dino = min(n_unfrozen, MAX_UNFROZEN_BLOCKS)
        n_conv = min((n_dino + 2) // 3, TOTAL_CONV_STAGES)   # 1 conv stage per ~3 DINO blocks
    
        for n, p in self.model.named_parameters():
            if "dino_slice_encoder.backbone" in n:
                m = re.search(r"\.layer\.(\d+)\.", n)
                if m:
                    p.requires_grad = int(m.group(1)) >= TOTAL_BACKBONE_BLOCKS - n_dino
                elif "layernorm" in n:
                    p.requires_grad = n_dino > 0
                else:
                    p.requires_grad = False
    
            elif "conv_slice_encoder.backbone" in n:
                m = re.search(r"\.stages\.(\d+)\.", n)
                if m:
                    p.requires_grad = int(m.group(1)) >= TOTAL_CONV_STAGES - n_conv
                elif "head.norm" in n:                        # final norm
                    p.requires_grad = n_conv > 0
                else:
                    p.requires_grad = False
    
        # Batch size: keep effective batch = 128
        if cycle >= 4:
            BATCH_SIZE = min(BATCH_SIZE, 2)
        elif cycle >= 2:
            BATCH_SIZE = min(BATCH_SIZE, 4)
        ACCUMULATION_STEP = 128 // BATCH_SIZE
        self.accumulation_steps = ACCUMULATION_STEP
        # NOTE: rebuild your DataLoader here if BATCH_SIZE changed
    
        print(f"[cycle {cycle}] dino: last {n_dino}/{TOTAL_BACKBONE_BLOCKS} layers | "
              f"conv: last {n_conv}/{TOTAL_CONV_STAGES} stages | "
              f"batch:{BATCH_SIZE}, acc:{ACCUMULATION_STEP}")


            
    def _save_checkpoint(self, path="checkpoint.pth"):
        ckpt = {
            'epoch': self.current_epoch,
            'cycle': self.current_cycle,
            'model': unwrap(self.model).state_dict(),
            'optimizer': self.optimizer.state_dict(),
            'scheduler': self.scheduler.state_dict(),
            'best_auc': self.best_auc,
            'patience': self.patience,
            'train_loss': self.train_loss,
            'val_loss': self.valid_loss,
            'valid_auc': self.valid_auc,
        }
        torch.save(ckpt, path)

    def load_checkpoint(self, path="/kaggle/input/rsna-knee-model/last_checkpoint.pth"):
        
        if not os.path.exists(path):
            print(f"No checkpoint found at {path}, starting fresh.")
            return
        ckpt = torch.load(path, map_location=self.device, weights_only=False) 
        unwrap(self.model).load_state_dict(ckpt['model'])
        self.optimizer.load_state_dict(ckpt['optimizer'])
        self.scheduler.load_state_dict(ckpt['scheduler'])
        self.best_auc = ckpt['best_auc']
        self.patience = ckpt['patience']
        self.train_loss = ckpt['train_loss']
        self.valid_loss = ckpt['val_loss']
        self.valid_auc = ckpt['valid_auc']
        print(f"Resume Previous chkpoit, cycle: {ckpt['cycle']}, epoch:{ckpt['epoch']}, patience:{self.patience}, best_auc = {self.best_auc}")
        self._unfreeze_slice_encoder(ckpt['cycle'])
            
        return ckpt['cycle'], ckpt['epoch']
        
    def _unpack(self, batch):
        modalities, meta, targets = batch
        modalities = [m.to(self.device, dtype=torch.float32) for m in modalities]
        meta    = meta.to(self.device, dtype=torch.float32)
        targets    = targets.to(self.device, dtype=torch.float32)
        return modalities, meta, targets
        


    def meta_trainer(self, drop_noise=False, verbose=True):
        """
        Metadata-only models, fitted on self.train, predicting self.val and self.GT_VAL.
        Output rows are in the SAME order as self.val / self.GT_VAL
        (so they line up with valid_loader / gt_valid_loader, which use shuffle=False).

        Saves:
          self.meta_preds[name] = {"val": (N_val, 12), "gt": (N_gt, 12)}   for LR / LGBM / HGB
          self.meta_name        = best model by macro AUC on self.val
          self.meta_val, self.meta_gt = predictions of that best model
          self.y_val_bin, self.y_gt_bin = binarized labels (N, 12)
        """
        global LIGHT_DF , LABELS
        noise_cols = ['InstanceNumber', 'Laterality_Clean_BILATERAL', 'Laterality_Clean_LEFT',
                      'Laterality_Clean_RIGHT', 'Laterality_Clean_Unknown'] if drop_noise else []
        
        meta_cols = ['ScanningSequence_SE', 'ScanningSequence_GR', 'ScanningSequence_IR', 'ScanningSequence_RM', 
                     'SequenceVariant_SK', 'SequenceVariant_SP', 'SequenceVariant_OSP', 'SequenceVariant_SS', 
                     'Manufacturer_Clean_Canon', 'Manufacturer_Clean_Fujifilm', 'Manufacturer_Clean_GE', 
                     'Manufacturer_Clean_Hitachi', 'Manufacturer_Clean_Philips', 'Manufacturer_Clean_Siemens',
                     'Manufacturer_Clean_Toshiba', 'Laterality_Clean_BILATERAL', 'Laterality_Clean_LEFT', 'Laterality_Clean_RIGHT', 
                     'Laterality_Clean_Unknown', 'PatientSex_Clean_Female', 'PatientSex_Clean_Male', 'PatientSex_Clean_Undefined',
                     'MRAcquisitionType_Clean_2D', 'MRAcquisitionType_Clean_3D', 'MRAcquisitionType_Clean_UNKNOWN', 'SliceNumberRate',
                     'InstanceNumber', 'Rows', 'Columns', 'MagneticFieldStrength', 'FlipAngle', 'Fluid_Sensitive', 'Fat_Suppression']   
        
        # one row per study: the 3 planes side by side
        wide = LIGHT_DF.pivot_table(index="StudyInstanceUID", columns="Anatomical_Plane",
                                    values=meta_cols, aggfunc="first")
        wide.columns = [f"{a}_{b}" for a, b in wide.columns]

        def feats(df_):      # metadata rows in the SAME order as df_ (NaN if a study is missing)
            return wide.reindex(df_["StudyInstanceUID"].values).values.astype(float)

        def labels(df_):
            return (df_[LABELS].values >= 0.5).astype(int)

        # GT studies may also sit inside self.train -> remove them when predicting GT
        gt_ids = set(self.GT_VAL["StudyInstanceUID"])
        train_no_gt = self.train[~self.train["StudyInstanceUID"].isin(gt_ids)]

        X_tr,  Y_tr  = feats(self.train),   labels(self.train)
        X_trg, Y_trg = feats(train_no_gt),  labels(train_no_gt)
        X_val, Y_val = feats(self.val),     labels(self.val)
        X_gt,  Y_gt  = feats(self.GT_VAL),  labels(self.GT_VAL)

        def prep(X_fit, X_other):    # fill NaN + scale, fitted on the training part only
            # pipe = make_pipeline(SimpleImputer(strategy="mean"), StandardScaler()).fit(X_fit)
            return X_fit , X_other #pipe.transform(X_fit), pipe.transform(X_other)

        Xa_tr, Xa_val = prep(X_tr, X_val)      # train -> val
        Xb_tr, Xb_gt = prep(X_trg, X_gt)       # train without GT -> GT

        factories = {
            "LR":   lambda: LogisticRegression(C=0.1, max_iter=1000),
            "LGBM": lambda: LGBMClassifier(n_estimators=100, learning_rate=0.05,
                                           random_state=42, verbose=-1),
            "HGB":  lambda: HistGradientBoostingClassifier(random_state=42, max_iter=100),
        }

        def auc_per_label(Y, P):
            out = []
            for j in range(Y.shape[1]):
                try:
                    out.append(roc_auc_score(Y[:, j], P[:, j]))
                except ValueError:             # only one class in this label
                    out.append(np.nan)
            return out

        n_lab = len(LABELS)
        self.meta_preds, val_auc, gt_auc = {}, {}, {}
        self.meta_models = {name: [] for name in factories}   # 12 fitted models per method
        self.meta_cols_used = list(wide.columns)              # exact feature order
        self.meta_wide = wide
        for name, make in factories.items():
            p_val = np.zeros((len(X_val), n_lab))
            p_gt = np.zeros((len(X_gt), n_lab))
            for j in range(n_lab):
                m_val = make().fit(Xa_tr, Y_tr[:, j])          # train -> val
                m_gt  = make().fit(Xb_tr, Y_trg[:, j])         # train without GT -> GT
                p_val[:, j] = m_val.predict_proba(Xa_val)[:, 1]
                p_gt[:, j]  = m_gt.predict_proba(Xb_gt)[:, 1]
                self.meta_models[name].append(m_val)           # keep the fitted model
            self.meta_preds[name] = {"val": p_val, "gt": p_gt}
            val_auc[name] = auc_per_label(Y_val, p_val)
            gt_auc[name] = auc_per_label(Y_gt, p_gt)

        if verbose:
            print("metadata features:", len(wide.columns), "| train:", len(X_tr),
                  "| val:", len(X_val), "| GT:", len(X_gt))
            for j, lab in enumerate(LABELS):
                row = " | ".join(f"{n}: val {val_auc[n][j]:.3f} GT {gt_auc[n][j]:.3f}"
                                 for n in factories)
                print(f"{lab:17s} {row}")
            print("-" * 90)
            for n in factories:
                print(f"{n:5s} macro  val AUC = {np.nanmean(val_auc[n]):.4f} | "
                      f"GT AUC = {np.nanmean(gt_auc[n]):.4f}")

        macro_val = {n: np.nanmean(val_auc[n]) for n in factories}
        self.meta_name = max(macro_val, key=macro_val.get)
        self.meta_val = self.meta_preds[self.meta_name]["val"]
        self.meta_gt = self.meta_preds[self.meta_name]["gt"]
        self.y_val_bin, self.y_gt_bin = Y_val, Y_gt
      
        print("best metadata model (by val):", self.meta_name)

      
    @torch.no_grad()
    def predict(self, ds):
        """Sigmoid probabilities (N, 12) in the same row order as ds (shuffle=False)."""
        loader = DataLoader(ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=4,
                            pin_memory=True, collate_fn=mri_collate_fn)
        self.model.eval()
        out = []
        for batch in tqdm(loader, desc="Predict"):
            modalities, meta, _ = self._unpack(batch)
            if self.scaler is not None:
                with torch.amp.autocast(device_type='cuda'):
                    logits = self.model(modalities, meta)
            else:
                logits = self.model(modalities, meta)
            out.append(torch.sigmoid(logits.float()).cpu().numpy())
        return np.vstack(out)
 
    def blend_search(self, ckpt_path="best_model_checkpoint.pth", **kwargs):
        """Load the best image checkpoint, predict val + GT, choose the blend weight once."""
        ckpt = torch.load(ckpt_path, map_location=self.device, weights_only=False)
        unwrap(self.model).load_state_dict(ckpt['model'])
 
        self.img_val = self.predict(self.val_ds)        # (N_val, 12)
        self.img_gt = self.predict(self.gt_val_ds)      # (N_gt, 12)
        print("image only   | val AUC: %.4f | GT AUC: %.4f"
              % (macro_auc(self.y_val_bin, self.img_val), macro_auc(self.y_gt_bin, self.img_gt)))
        print("metadata only| val AUC: %.4f | GT AUC: %.4f  (%s)"
              % (macro_auc(self.y_val_bin, self.meta_val), macro_auc(self.y_gt_bin, self.meta_gt),
                 self.meta_name))
 
        res = search_blend_weight(self.img_val, self.meta_val, self.y_val_bin,
                                  self.img_gt, self.meta_gt, self.y_gt_bin, **kwargs)
        self.best_w = res["final_w"]
        return res
        
    def log_blend(self, epoch, weights = [0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0]):
        """Blended macro AUC for a few fixed weights, using this epoch's image predictions."""
        img_val, img_gt = self.epoch_preds["val"], self.epoch_preds["gt"]
        row = {"epoch": epoch}
        for w in weights:
            row[f"val_{w}"] = macro_auc(self.y_val_bin, blend(img_val, self.meta_val, w))
            row[f"gt_{w}"] = macro_auc(self.y_gt_bin, blend(img_gt, self.meta_gt, w))
        self.blend_log.append(row)
 
        print("blend weight w : " + "  ".join(f"{w:>6}" for w in weights))
        print("      val AUC  : " + "  ".join(f"{row[f'val_{w}']:.4f}" for w in weights))
        print("      GT  AUC  : " + "  ".join(f"{row[f'gt_{w}']:.4f}" for w in weights))
        print(f"      gain on val (best w vs w=0): "
              f"{max(row[f'val_{w}'] for w in weights) - row['val_0.0']:+.4f}")    
        
    def log_label_weights(self, epoch, **kwargs):
        """Fit per-label weights on this epoch's val predictions and print the result."""
        if not hasattr(self, "label_w_log"):
            self.label_w_log = []
        res = fit_label_weights(self.epoch_preds["val"], self.meta_val, self.y_val_bin,
                                self.epoch_preds["gt"], self.meta_gt, self.y_gt_bin, **kwargs)
        res["epoch"] = epoch
        self.label_w_log.append(res)
        print_label_weights(res, epoch)
        return res    
        
    def train_one_epoch_accumulation(self, epoch):
        self.model.train()
        running_loss = 0.0
        all_preds, all_targets = [], []
    
        progress_bar = tqdm(self.train_loader, desc=f"Epoch {epoch} [Train]")
        self.optimizer.zero_grad()
    
        for i, batch in enumerate(progress_bar):
            modalities, meta, targets = self._unpack(batch)
            if i == 0:
                print(len(modalities), modalities[0].shape)
    
            if self.scaler is not None:
                with torch.amp.autocast(device_type='cuda'):
                    outputs = self.model(modalities, meta)
                    loss = self.criterion(outputs, targets)
                    loss = loss / self.accumulation_steps
                self.scaler.scale(loss).backward()
            else:
                outputs = self.model(modalities, meta)
                loss = self.criterion(outputs, targets)
                loss = loss / self.accumulation_steps
                loss.backward()
    
            if (i + 1) % self.accumulation_steps == 0 or (i + 1) == len(self.train_loader):
                if self.scaler is not None:
                    self.scaler.step(self.optimizer)
                    self.scaler.update()
                else:
                    self.optimizer.step()
                self.optimizer.zero_grad()
    
            running_loss += loss.item() * self.accumulation_steps * targets.size(0)
    
            # detach + float, so numpy works and fp16 is handled
            all_preds.append(torch.sigmoid(outputs.detach().float()).cpu().numpy())
            all_targets.append(targets.detach().cpu().numpy())
            progress_bar.set_postfix({'loss': f"{loss.item() * self.accumulation_steps:.4f}"})
    
        epoch_loss = running_loss / len(self.train_loader.dataset)
    
        all_preds = np.vstack(all_preds)        # <-- missing before
        all_targets = np.vstack(all_targets)    # <-- missing before
    
        auc_scores = []
        for j in range(all_targets.shape[1]):
            y_true = all_targets[:, j]
            y_pred = all_preds[:, j]
            if not np.array_equal(y_true, y_true.astype(bool)):
                y_true = (y_true >= 0.5).astype(int)
            try:
                auc_scores.append(roc_auc_score(y_true, y_pred))
            except ValueError:
                auc_scores.append(np.nan)
    
        epoch_auc = np.nanmean(auc_scores)
        return epoch_loss, epoch_auc

    def valid_one_epoch(self, epoch , data_loader, tag=None):
        self.model.eval()
        running_loss = 0.0
        all_preds, all_targets = [], []
    
        progress_bar = tqdm(data_loader, desc=f"Epoch {epoch} [Valid]")
    
        with torch.no_grad():
            for batch in progress_bar:
                modalities, meta, targets = self._unpack(batch)
    
                if self.scaler is not None:
                    with torch.amp.autocast(device_type='cuda'):
                        outputs = self.model(modalities, meta)
                        loss    = self.criterion(outputs, targets)
                else:
                    outputs = self.model(modalities, meta)
                    loss    = self.criterion(outputs, targets)
    
                running_loss += loss.item() * targets.size(0)
                
                # Apply sigmoid to convert logits to probabilities [0, 1]
                preds = torch.sigmoid(outputs).cpu().numpy()
                all_preds.append(preds)
                all_targets.append(targets.cpu().numpy())
    
        # Combine batches into 2D arrays of shape (N_samples, 12)
        all_preds = np.vstack(all_preds)
        all_targets = np.vstack(all_targets)
        epoch_loss = running_loss / len(data_loader.dataset)
    
        # Compute Macro ROC-AUC across all 12 targets safely
        num_targets = all_targets.shape[1]
        auc_scores = []
    
        for i in range(num_targets):
            y_true = all_targets[:, i]
            y_pred = all_preds[:, i]
    
            # Binarize soft labels if validation set contains continuous targets
            if not np.array_equal(y_true, y_true.astype(bool)):
                y_true = (y_true >= 0.5).astype(int)
    
            try:
                auc = roc_auc_score(y_true, y_pred)
                auc_scores.append(auc)
            except ValueError:
                # Triggered if a validation fold contains only 0s or only 1s for target i
                auc_scores.append(np.nan)
    
        # Average valid target scores matching the competition metric formula[cite: 1]
        epoch_auc = np.nanmean(auc_scores)
        if tag is not None:               
            self.epoch_preds[tag] = all_preds
        return epoch_loss, epoch_auc        

    def training(self):
        global IMAGE_SIZE, BATCH_SIZE, ACCUMULATION_STEP  
        print(f"Starting Training on device: {self.device}")        
        # self.meta_trainer(drop_noise=False, verbose=True)
        def make_chunks(df, n_chunks, seed=None):
            shuffled = df.sample(frac=1, random_state=seed).reset_index(drop=True)
            chunks = np.array_split(shuffled, n_chunks)
            return chunks
    
        num_cycles = NUM_CYCLE
        n_chunks = NUM_CHUNKS

        resume_chunk = self.resume_epoch - (self.resume_cycle * n_chunks) + 1
        if resume_chunk >= n_chunks:
            self.resume_cycle += 1
            resume_chunk = 0
        resume_chunk = max(0, resume_chunk)
        
        for cycle in range(self.resume_cycle, num_cycles):   # start from saved cycle, not 0
            self.current_cycle = cycle
            chunks = make_chunks(self.train, n_chunks=n_chunks, seed=cycle)
            stop_training = False
            self._unfreeze_slice_encoder(cycle)
        
            # if cycle >= 16:                       # every cycle, not only 16
            #     IMAGE_SIZE, BATCH_SIZE, ACCUMULATION_STEP = 384, 2, 64
            #     self.accumulation_steps = ACCUMULATION_STEP
            # else:
            #     self._unfreeze_slice_encoder(cycle)
                
            # if cycle == 16:   
            #     cleanup("before 384px")
            #     self.optimizer.param_groups[0]["lr"] = 5e-6
            #     self.optimizer.param_groups[1]["lr"] = 5e-5
            #     self.scheduler = CosineAnnealingLR(self.optimizer, T_max=17, eta_min=5e-7)
            #     self.best_auc, self.patience = 0.0, 0
            
            self.val_ds = KneeDataset(train_labeled=self.val, series_merged=LIGHT_DF, split='train', augmentation=False)
            self.valid_loader = DataLoader(
                dataset=self.val_ds, batch_size=BATCH_SIZE, shuffle=False,
                num_workers=2, pin_memory=True, collate_fn=mri_collate_fn
            )
            self.gt_val_ds = KneeDataset(train_labeled=self.GT_VAL, series_merged=LIGHT_DF, split='train', augmentation=False)
            self.gt_valid_loader = DataLoader(
                dataset=self.gt_val_ds, batch_size=BATCH_SIZE, shuffle=False,
                num_workers=2, pin_memory=True, collate_fn=mri_collate_fn
            )
            
            for i in range(resume_chunk , n_chunks):
                epoch = cycle * n_chunks + i
                
                self.current_epoch = epoch
                chunk_df = chunks[i].reset_index(drop=True)
                train_ds = KneeDataset(train_labeled=chunk_df, series_merged=LIGHT_DF,
                                       split='train', augmentation=True)
                self.train_loader = DataLoader(
                    dataset=train_ds, batch_size=BATCH_SIZE, shuffle=True,
                    num_workers=4, pin_memory=True, collate_fn=mri_collate_fn
                )
    
                print(f"Epoch {epoch} — cycle {cycle}, chunk {i}, size {len(chunk_df)},  image_size = {IMAGE_SIZE} | batch = {BATCH_SIZE} | accumulation ={ACCUMULATION_STEP}|LR:{self.optimizer.param_groups[0]['lr']:.6f}")
                tr_loss , tr_auc= self.train_one_epoch_accumulation(epoch)
                val_loss, val_auc = self.valid_one_epoch(epoch , self.valid_loader, tag="val")
                gt_val_loss, gt_val_auc = self.valid_one_epoch(epoch , self.gt_valid_loader, tag="gt")
                self.log_blend(epoch)
                self.log_label_weights(epoch)
                self.train_loss.append(tr_loss)
                self.valid_loss.append(val_loss)
                self.valid_auc.append(val_auc)
                
                print(f"\nEpoch {epoch} Summary: Train Loss = {tr_loss:.4f} | Train AUC = {tr_auc:.4f} | "
                      f"Valid Loss = {val_loss:.4f} | Valid AUC = {val_auc:.4f} | "
                      f"GT Valid Loss = {gt_val_loss:.4f} | GT Valid AUC = {gt_val_auc:.4f} | "
                      f"LR0:{self.optimizer.param_groups[0]['lr']:.6f} |  LR1:{self.optimizer.param_groups[1]['lr']:.6f}\n")
                
                if epoch > WARMUP_EPOCHS and (val_auc - self.best_auc) < MIN_DELTA:
                    self.stall += 1
                else:
                    self.stall = 0
                    
                if epoch <= WARMUP_EPOCHS:
                    if val_auc > self.best_auc:
                        print(f"Validation AUC improved from {self.best_auc:.4f} to {val_auc:.4f}. Saving best_model.pth...\n")
                        self.best_auc = val_auc
                        self.patience = 0
                        self._save_checkpoint( "best_model_checkpoint.pth")
                        
                else:
                    if val_auc > self.best_auc:
                        print(f"Validation AUC improved from {self.best_auc:.4f} to {val_auc:.4f}. Saving best_model.pth...\n")
                        self.best_auc = val_auc
                        self.patience = 0
                        self._save_checkpoint("best_model_checkpoint.pth")
                        
                    else:
                        self.patience += 1
                        
                self._save_checkpoint( "last_checkpoint.pth")
                
                if self.patience >= PATIENCE or self.stall >= PATIENCE:
                    stop_training = True
                    break

                del self.train_loader, train_ds, chunk_df
                cleanup(f"epoch {epoch}")
                    
            self.scheduler.step() 
            resume_chunk = 0
            if hasattr(self, 'valid_loader'):
                del self.valid_loader
            if hasattr(self, 'gt_valid_loader'):
                del self.gt_valid_loader
            if hasattr(self, 'train_loader'):
                del self.train_loader
            gc.collect()
            cleanup(f"cycle {cycle}")
            if stop_training:
                break
    
        plot_loss(self.valid_loss, self.train_loss, self.valid_auc, 'loss_trend.png')
        self.blend_search()
        res = fit_label_weights(self.img_val, self.meta_val, self.y_val_bin,
                            self.img_gt,  self.meta_gt,  self.y_gt_bin)
        print_label_weights(res, epoch)
        
        print(f"Training Complete. Best Validation AUC: {self.best_auc:.4f}")  


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
trainer=TrainerSingleFold(train_df= TRAIN_LABELD ,epochs = (NUM_CYCLE * NUM_CHUNKS), device = device, scaler = GradScaler() )
# start_mem_watcher()
trainer.meta_trainer(drop_noise=False, verbose=True)
# trainer.training()


num metadata features: 33


Loading weights:   0%|          | 0/223 [00:00<?, ?it/s]

Selected Encoder: dinov2_small
Selected Encoder: convnext_nano
Total Parameters: 14,952,560
Trainable Parameters: 14,952,560
Trainable Parameters: 0
No checkpoint found at /kaggle/input/rsna-knee-model/last_checkpoint.pth, starting fresh.
metadata features: 99 | train: 3914 | val: 435 | GT: 58
ACL               LR: val 0.681 GT 0.763 | LGBM: val 0.690 GT 0.765 | HGB: val 0.688 GT 0.790
MCL               LR: val 0.617 GT 0.624 | LGBM: val 0.614 GT 0.626 | HGB: val 0.595 GT 0.644
Medial Meniscus   LR: val 0.545 GT 0.448 | LGBM: val 0.553 GT 0.602 | HGB: val 0.547 GT 0.580
Lateral Meniscus  LR: val 0.535 GT 0.681 | LGBM: val 0.495 GT 0.708 | HGB: val 0.509 GT 0.684
Medial OA         LR: val 0.599 GT 0.739 | LGBM: val 0.616 GT 0.780 | HGB: val 0.607 GT 0.736
Lateral OA        LR: val 0.667 GT 0.668 | LGBM: val 0.689 GT 0.710 | HGB: val 0.687 GT 0.649
PF OA             LR: val 0.626 GT 0.676 | LGBM: val 0.618 GT 0.699 | HGB: val 0.606 GT 0.681
Effusion          LR: val 0.647 GT 0.820 | LGBM

# ensemble of dino+conv+meta on val data

In [15]:
# IMAGE_SIZE = 384
# BATCH_SIZE = 16
# train_df= TRAIN_LABELD.copy()
# train, val        = train_test_split(train_df, test_size=0.1, random_state=42)
# val   = val.reset_index(drop=True)
# GT_VAL   = GT_VAL.reset_index(drop=True) #58 Ground Truth samples

# val_ds   = KneeDataset(train_labeled =  val , series_merged = LIGHT_DF, split='train', augmentation=False)
# gt_val_ds   = KneeDataset(train_labeled =  GT_VAL , series_merged = LIGHT_DF, split='train', augmentation=False)
# valid_loader = DataLoader(
#                 dataset=val_ds, batch_size=BATCH_SIZE, shuffle=False,
#                 num_workers=2, pin_memory=True, collate_fn=mri_collate_fn
#             )
# gt_valid_loader = DataLoader(
#                 dataset= gt_val_ds, batch_size=BATCH_SIZE, shuffle=False,
#                 num_workers=2, pin_memory=True, collate_fn=mri_collate_fn
#             )

# combined_model = MRI25DFusionModelViewRoutingDualEncoder(
#             d=256, nhead=4, num_layers=2,
#             num_modalities=3, max_slices=NUM_SLICES,
#             num_metadata_features=33, d_meta=32
#         )
# path = '/kaggle/input/datasets/malihealavi/rsna-knee-model/best_model_checkpoint.pth'
# ckpt = torch.load(path, map_location='cpu', weights_only=False) 
# combined_model.load_state_dict(ckpt['model'])

# ENCODER_BB = 'convnext_nano'
# conv_model = MRI25DFusionModelViewRoutingSingleEncoder(
#             d=256, nhead=4, num_layers=2,
#             num_modalities=3, max_slices=NUM_SLICES,
#             num_metadata_features=33, d_meta=32
#         )
# path = '/kaggle/input/datasets/malihealavi/knee-final-models/conv_model.pth'
# ckpt = torch.load(path, map_location='cpu', weights_only=False) 
# conv_model.load_state_dict(ckpt)

# ENCODER_BB = 'dinov2_small'
# dino_model = MRI25DFusionModelViewRoutingSingleEncoder(
#             d=256, nhead=4, num_layers=2,
#             num_modalities=3, max_slices=NUM_SLICES,
#             num_metadata_features=33, d_meta=32
#         )
# path = '/kaggle/input/datasets/malihealavi/knee-final-models/dino_model.pth'
# ckpt = torch.load(path, map_location='cpu', weights_only=False) 
# dino_model.load_state_dict(ckpt)


# def _unpack( batch):
#     device = 'cuda' if torch.cuda.is_available() else 'cpu'
    
#     modalities, meta, targets = batch
#     modalities = [m.to(device, dtype=torch.float32) for m in modalities]
#     meta    = meta.to(device, dtype=torch.float32)
#     targets    = targets.to(device, dtype=torch.float32)
#     return modalities, meta, targets
       

# # -----------------------------
# import numpy as np, torch
# from sklearn.metrics import roc_auc_score
# from tqdm import tqdm

# device = 'cuda' if torch.cuda.is_available() else 'cpu'

# @torch.no_grad()
# def get_preds(model, loader):
#     model.to(device).eval()
#     preds, targets_all = [], []
#     for batch in tqdm(loader):
#         modalities, meta, targets = _unpack(batch)
#         with torch.amp.autocast(device_type='cuda', enabled=(device == 'cuda')):
#             outputs = model(modalities, meta)
#         preds.append(torch.sigmoid(outputs.float()).cpu().numpy())
#         targets_all.append(targets.cpu().numpy())
#     return np.vstack(preds), np.vstack(targets_all)

# def macro_auc(y_true_all, y_pred_all):
#     scores = []
#     for i in range(y_true_all.shape[1]):
#         y_true = y_true_all[:, i]
#         if not np.array_equal(y_true, y_true.astype(bool)):
#             y_true = (y_true >= 0.5).astype(int)
#         try:
#             scores.append(roc_auc_score(y_true, y_pred_all[:, i]))
#         except ValueError:
#             scores.append(np.nan)
#     return np.nanmean(scores)

# # Get predictions from each model
# conv_preds, targets = get_preds(conv_model, valid_loader)
# dino_preds, _       = get_preds(dino_model, valid_loader)

# # Ensemble = average of probabilities
# ens_preds = (conv_preds + dino_preds) / 2

# print("ConvNeXt AUC:", macro_auc(targets, conv_preds))
# print("DINO AUC    :", macro_auc(targets, dino_preds))
# print("Ensemble AUC:", macro_auc(targets, ens_preds))

# ----------------meta data--------------

# 1) Image predictions from both models (same loader order, shuffle=False)
# conv_val, _ = get_preds(conv_model, valid_loader)
# dino_val, _ = get_preds(dino_model, valid_loader)
# conv_gt,  _ = get_preds(conv_model, gt_valid_loader)
# dino_gt,  _ = get_preds(dino_model, gt_valid_loader)

# # # 2) Image ensemble
# img_val = (conv_val + dino_val) / 2
# img_gt  = (conv_gt  + dino_gt)  / 2

# # 3) Metadata predictions (already saved in your trainer)
# meta_val, meta_gt = trainer.meta_val, trainer.meta_gt
# y_val, y_gt = trainer.y_val_bin, trainer.y_gt_bin

# # 4) Find the best blend weight on val, then check on GT
# for w in np.arange(0,1,0.1):
#     v = macro_auc(y_val, blend(img_val, meta_val, w))
#     g = macro_auc(y_gt,  blend(img_gt,  meta_gt,  w))
#     print(f"w={w}  val AUC={v:.4f}  GT AUC={g:.4f}")

# Submission

In [16]:
NUM_SLICES = 16
IMAGE_SIZE = 256
BATCH_SIZE = 16
ENCODER_BB = 'dinov2_small'

import os
import ast
import pandas as pd
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm

TARGET_COLS = ['ACL', 'MCL', 'Medial Meniscus', 'Lateral Meniscus', 'Medial OA',
               'Lateral OA', 'PF OA', 'Effusion', 'Synovitis', "Baker's", "Contusion", 'Fracture']


def pick_best_series(group):
    best = group[(group.Fluid_Sensitive == True) & (group.Fat_Suppression == True)]
    if len(best) == 0:
        best = group[group.Fluid_Sensitive == True]   # fallback: fluid-sensitive only
    if len(best) == 0:
        best = group                                  # fallback: take whatever exists
    return best.iloc[0]

def unwrap(m):
    """DataParallel stores the real model under `.module` — use this
    whenever you need to reach a custom attribute (e.g. .backbone)."""
    return m.module if isinstance(m, nn.DataParallel) else m

def _load_model(checkpoint_path, device, d=256, nhead=4, num_layers=2,
                num_modalities=3, max_slices=NUM_SLICES):

    model = MRI25DFusionModelViewRouting(d=d, nhead=nhead, num_layers=num_layers,num_modalities=num_modalities, max_slices=max_slices)
    
    ckpt = torch.load(checkpoint_path, map_location=device, weights_only=False) 
    
    state_dict = {k.replace('module.', ''): v for k, v in ckpt['model'].items()}
    unwrap(model).load_state_dict(state_dict)

    model.to(device)
    if torch.cuda.device_count() > 1:
        print(f"Using {torch.cuda.device_count()} GPUs")
        model = nn.DataParallel(model)
        
    model.eval()
    return model


class SubmissionDataset(Dataset):
    """Wraps KneeDataset (split='test') and also returns each item's StudyInstanceUID,
    so predictions get matched back to studies by ID, never by position."""
    def __init__(self, test_df, test_light_df):
        self.base = KneeDataset(train_labeled=test_df, series_merged=test_light_df,
                                 split='test', augmentation=False)
        self.study_uids = test_df['StudyInstanceUID'].tolist()

    def __len__(self):
        return len(self.base)

    def __getitem__(self, idx):
        return self.base[idx], self.study_uids[idx]


def mri_collate_fn_submit(batch):
    # batch items are: ((list_3d, meta), study_uid)
    list_3ds = [item[0][0] for item in batch]
    metas = [item[0][1] for item in batch]
    uids_out = [item[1] for item in batch]
    
    num_modalities = len(list_3ds[0])
    
    modalities_out = [
        torch.stack([list_3d[m] for list_3d in list_3ds], dim=0)
        for m in range(num_modalities)
    ]
    meta_out = torch.stack(metas, dim=0)
    
    return modalities_out, meta_out, uids_out


def submit(test_df, test_series_df, checkpoint_path, device,
           batch_size=BATCH_SIZE, num_workers=4, out_path='submission.csv'):

    if 'file_path' not in test_series_df.columns:
        raise ValueError("test_series_df needs a 'file_path' column (DICOM paths per series), "
                          "same as SERIES_MERGED — pick_best_series/KneeDataset both need it.")

    # 1. one best series per (study, plane) — same selection logic used in training
    test_light_df = (
        test_series_df
        .groupby(['StudyInstanceUID', 'Anatomical_Plane'])
        .apply(pick_best_series)
        .reset_index(drop=True)
    )
    def has_valid_files(x):
        if not isinstance(x, str):
            return False
        try:
            return len(ast.literal_eval(x)) > 0
        except Exception:
            return False
 
    test_light_df = test_light_df[test_light_df['file_path'].apply(has_valid_files)]

    # 2. studies missing a plane would crash the batch collate later — pull them out
    #    now and give them the neutral 0.5 score instead
    planes_per_study = test_light_df.groupby('StudyInstanceUID')['Anatomical_Plane'].apply(set)
    complete_uids = set(planes_per_study[planes_per_study.apply(
        lambda s: {'Axial', 'Coronal', 'Sagittal'}.issubset(s))].index)
    incomplete_uids = set(test_df['StudyInstanceUID']) - complete_uids
    if incomplete_uids:
        print(f"WARNING: {len(incomplete_uids)} studies missing a plane — "
              f"filling with neutral 0.5 scores: {list(incomplete_uids)[:5]}")

    good_df = test_df[test_df['StudyInstanceUID'].isin(complete_uids)].reset_index(drop=True)

    # --------EXTRACT META 
    sample_df = test_light_df[['Manufacturer',
                    'Laterality',
                    'PatientSex',
                     'ScanningSequence',
                    'SequenceVariant',
                    'MRAcquisitionType']].copy()
    # Run cleaning and encoding
    encoded_df = clean_and_encode_dicom_metadata(sample_df)
    encoded_df['SliceNumberRate'] = NUM_SLICES/test_light_df['InstanceNumber']
    encoded_df['InstanceNumber'] = test_light_df['InstanceNumber']
    encoded_df['Rows'] = np.log(test_light_df['Rows'])
    encoded_df['Columns'] = np.log(test_light_df['Columns'])
    encoded_df[[ 'MagneticFieldStrength','FlipAngle' , 'Fluid_Sensitive', 'Fat_Suppression'	]] = test_light_df[[ 'MagneticFieldStrength','FlipAngle' , 'Fluid_Sensitive', 'Fat_Suppression'	]]
    print("Encoded Feature Columns:")
    
    num_cols = ['InstanceNumber', 'Rows', 'Columns', 'MagneticFieldStrength',
                'FlipAngle', 'SliceNumberRate']
    
    # scaler = MinMaxScaler()
    # encoded_df[num_cols] = scaler.fit_transform(encoded_df[num_cols])
    encoded_df[num_cols] = std_scaler.transform(encoded_df[num_cols])
    
    # Impute AFTER scaling so missing becomes an out-of-range sentinel
    encoded_df[num_cols] = encoded_df[num_cols].fillna(-1)
    encoded_df [['StudyInstanceUID','SeriesInstanceUID','file_path', 'Anatomical_Plane']] = test_light_df[['StudyInstanceUID','SeriesInstanceUID','file_path', 'Anatomical_Plane']]
    test_light_df = None
    test_light_df = encoded_df.copy()
    
    
    # 3. load model
    model = _load_model(checkpoint_path, device)

    # 4. run inference
    ds = SubmissionDataset(good_df, test_light_df)
    loader = DataLoader(ds, batch_size=batch_size, shuffle=False,
                         num_workers=num_workers, pin_memory=True, drop_last = False,
                         collate_fn=mri_collate_fn_submit)

    all_uids, all_preds = [], []
    with torch.no_grad():
        for modalities, meta, uids in tqdm(loader, desc="Predicting"):
            modalities = [m.to(device, dtype=torch.float32) for m in modalities]
            meta = meta.to(device, dtype=torch.float32)
            
            if device.type == 'cuda':
                with torch.amp.autocast(device_type='cuda'):
                    outputs = model(modalities, meta)
            else:
                outputs = model(modalities, meta)
                
            preds = torch.sigmoid(outputs).float().cpu().numpy()
            all_preds.append(preds)
            all_uids.extend(uids)

    all_preds = np.vstack(all_preds) if all_preds else np.empty((0, len(TARGET_COLS)))
    pred_df = pd.DataFrame(all_preds, columns=TARGET_COLS)
    pred_df.insert(0, 'StudyInstanceUID', all_uids)

    # 5. add neutral-score rows for the incomplete studies
    if incomplete_uids:
        fallback_rows = pd.DataFrame({'StudyInstanceUID': list(incomplete_uids),
                                       **{col: 0.5 for col in TARGET_COLS}})
        pred_df = pd.concat([pred_df, fallback_rows], ignore_index=True)

    # 6. re-align to test_df — guarantees exactly one row per StudyInstanceUID, matched by ID
    submission = test_df[['StudyInstanceUID']].merge(pred_df, on='StudyInstanceUID', how='left')
    submission[TARGET_COLS] = submission[TARGET_COLS].fillna(0.5)  # safety net

    submission.to_csv(out_path, index=False)
    print(f"Saved {len(submission)} rows to {out_path}")
    return submission


In [17]:
import ast, traceback
import numpy as np, pandas as pd, torch
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm

META_COLS = ['ScanningSequence_SE','ScanningSequence_GR','ScanningSequence_IR','ScanningSequence_RM',
             'SequenceVariant_SK','SequenceVariant_SP','SequenceVariant_OSP','SequenceVariant_SS',
             'Manufacturer_Clean_Canon','Manufacturer_Clean_Fujifilm','Manufacturer_Clean_GE',
             'Manufacturer_Clean_Hitachi','Manufacturer_Clean_Philips','Manufacturer_Clean_Siemens',
             'Manufacturer_Clean_Toshiba','Laterality_Clean_BILATERAL','Laterality_Clean_LEFT',
             'Laterality_Clean_RIGHT','Laterality_Clean_Unknown','PatientSex_Clean_Female',
             'PatientSex_Clean_Male','PatientSex_Clean_Undefined','MRAcquisitionType_Clean_2D',
             'MRAcquisitionType_Clean_3D','MRAcquisitionType_Clean_UNKNOWN','SliceNumberRate',
             'InstanceNumber','Rows','Columns','MagneticFieldStrength','FlipAngle',
             'Fluid_Sensitive','Fat_Suppression']
NUM_COLS = ['InstanceNumber','Rows','Columns','MagneticFieldStrength','FlipAngle','SliceNumberRate']
RAW_CAT  = ['Manufacturer','Laterality','PatientSex','ScanningSequence','SequenceVariant','MRAcquisitionType']
TARGET_COLS = ['ACL', 'MCL', 'Medial Meniscus', 'Lateral Meniscus', 'Medial OA',
               'Lateral OA', 'PF OA', 'Effusion', 'Synovitis', "Baker's", "Contusion", 'Fracture']


class SafeSubmissionDataset(Dataset):
    """If one study fails to load, return None instead of crashing."""
    def __init__(self, test_df, test_light_df, meta_ref=None):
        self.uids = test_df['StudyInstanceUID'].tolist()
        self.base = KneeDataset(train_labeled=test_df, series_merged=test_light_df,
                                split='test', augmentation=False, meta_ref=meta_ref)
    def __len__(self):
        return len(self.uids)
    def __getitem__(self, idx):
        try:
            return self.base[idx], self.uids[idx]
        except Exception:
            return None, self.uids[idx]


def safe_collate(batch):
    good = [b for b in batch if b[0] is not None]
    if not good:
        return None
    return mri_collate_fn_submit(good)


def submit_ensemble(test_df, test_series_df, models, trainer, best_w, device,
                    batch_size=BATCH_SIZE, num_workers=2, out_path='submission.csv',
                    meta_ref=None , deadline=float('inf')):
    device = torch.device(device)
    errors = []

    def note(where, e):
        msg = f"[{where}] {type(e).__name__}: {e}"
        errors.append(msg)
        print("WARNING", msg)
        traceback.print_exc()

    uids_all = test_df['StudyInstanceUID'].drop_duplicates().tolist()

    def write(pred_df):
        pred_df = pred_df.drop_duplicates('StudyInstanceUID')
        sub = test_df[['StudyInstanceUID']].merge(pred_df, on='StudyInstanceUID', how='left')
        for c in TARGET_COLS:
            if c not in sub.columns:
                sub[c] = 0.5
        sub[TARGET_COLS] = (sub[TARGET_COLS].apply(pd.to_numeric, errors='coerce')
                            .fillna(0.5).clip(0, 1))
        sub.to_csv(out_path, index=False)
        print(f"Saved {len(sub)} rows to {out_path} | errors: {len(errors)}")
        return sub

    def has_valid_files(x):
        if not isinstance(x, str):
            return False
        try:
            return len(ast.literal_eval(x)) > 0
        except Exception:
            return False

    def core():
        # ---------- 1. one best series per (study, plane), no groupby.apply
        df = test_series_df.copy()
        for c in ['StudyInstanceUID','SeriesInstanceUID','Anatomical_Plane','file_path'] + RAW_CAT + \
                 ['InstanceNumber','Rows','Columns','MagneticFieldStrength','FlipAngle']:
            if c not in df.columns:
                df[c] = np.nan
        for c in ['Fluid_Sensitive', 'Fat_Suppression']:
            if c not in df.columns:
                df[c] = False
        fluid = (df['Fluid_Sensitive'] == True)
        fat = (df['Fat_Suppression'] == True)
        df['_prio'] = fluid.astype(int) + (fluid & fat).astype(int)   # 2 = best, 1 = fluid only, 0 = other
        df = df[df['file_path'].apply(has_valid_files)]
        df = (df.sort_values('_prio', ascending=False, kind='stable')
                .drop_duplicates(['StudyInstanceUID', 'Anatomical_Plane'])
                .reset_index(drop=True))
        if len(df) == 0:
            raise ValueError("no valid series in test_series_df")

        planes = df.groupby('StudyInstanceUID')['Anatomical_Plane'].apply(set)
        complete = set(planes[planes.apply(lambda s: {'Axial','Coronal','Sagittal'}.issubset(s))].index)
        incomplete = set(uids_all) - complete
        if incomplete:
            print(f"WARNING: {len(incomplete)} studies missing a plane -> 0.5")
        good_df = (test_df.drop_duplicates('StudyInstanceUID')
                   [lambda d: d['StudyInstanceUID'].isin(complete)].reset_index(drop=True))
        if len(good_df) == 0:
            raise ValueError("no complete studies")
        good_uids = good_df['StudyInstanceUID'].tolist()

        # ---------- 2. metadata features (safe)
        try:
            enc = clean_and_encode_dicom_metadata(df[RAW_CAT].copy())
            if len(enc) == len(df):
                enc.index = df.index
        except Exception as e:
            note("clean_and_encode", e)
            enc = pd.DataFrame(index=df.index)

        num = lambda c: pd.to_numeric(df[c], errors='coerce').replace([np.inf, -np.inf], np.nan)
        inst, rows, cols_ = num('InstanceNumber'), num('Rows'), num('Columns')
        enc['InstanceNumber'] = inst
        enc['SliceNumberRate'] = NUM_SLICES / inst.where(inst > 0)
        enc['Rows'] = np.log(rows.where(rows > 0))
        enc['Columns'] = np.log(cols_.where(cols_ > 0))
        enc['MagneticFieldStrength'] = num('MagneticFieldStrength')
        enc['FlipAngle'] = num('FlipAngle')
        for c in ['Fluid_Sensitive', 'Fat_Suppression']:
            enc[c] = df[c].map({True: 1.0, False: 0.0})

        # same column order as training; missing columns are added
        order = [c for c in LIGHT_DF.columns if c in set(META_COLS)]
        if len(order) != len(META_COLS):
            order = META_COLS
        enc = enc.reindex(columns=order).replace([np.inf, -np.inf], np.nan).astype(float)

        other = [c for c in enc.columns if c not in NUM_COLS]
        enc[other] = enc[other].fillna(0)
        try:
            enc[NUM_COLS] = std_scaler.transform(enc[NUM_COLS])
        except Exception as e:
            note("std_scaler", e)
        enc[NUM_COLS] = enc[NUM_COLS].replace([np.inf, -np.inf], np.nan).fillna(-1)

        id_cols = ['StudyInstanceUID', 'SeriesInstanceUID', 'file_path', 'Anatomical_Plane']

        # ---------- 3. metadata model predictions (safe)
        print('Start of metadata model predictions')
        print(enc.columns)
        meta_pred = None
        try:
            tmp = enc.copy()
            tmp[['StudyInstanceUID', 'Anatomical_Plane']] = df[['StudyInstanceUID', 'Anatomical_Plane']]
            wide = tmp.pivot_table(index='StudyInstanceUID', columns='Anatomical_Plane',
                                   values=order, aggfunc='first')
            wide.columns = [f"{a}_{b}" for a, b in wide.columns]
            X = wide.reindex(index=good_uids, columns=trainer.meta_cols_used).to_numpy(dtype=float)
            print("NaN in meta features:", int(np.isnan(X).sum()))
            mm = trainer.meta_models[trainer.meta_name]
            try:
                meta_pred = np.column_stack([m.predict_proba(X)[:, 1] for m in mm])
            except Exception:
                X2 = np.nan_to_num(X, nan=-1)
                meta_pred = np.column_stack([m.predict_proba(X2)[:, 1] for m in mm])
            if meta_pred.shape != (len(good_uids), len(TARGET_COLS)):
                raise ValueError(f"bad meta_pred shape {meta_pred.shape}")
        except Exception as e:
            note("metadata models", e)
            meta_pred = None
        print('End of metadata model predictions')

        # ---------- 4. image models (safe)
        print('Start of image model predictions')
        img_pred = np.full((len(good_uids), len(TARGET_COLS)), np.nan)
        try:
            gc.collect()
            if device.type == 'cuda':
                torch.cuda.empty_cache()
            enc[id_cols] = df[id_cols]
            light_df = enc.copy()
            for m in models:
                m.to(device).eval()
            ds = SafeSubmissionDataset(good_df, light_df, meta_ref=meta_ref)
            loader = DataLoader(ds, batch_size=batch_size, shuffle=False, num_workers=num_workers,
                                pin_memory=(device.type == 'cuda'), drop_last=False,
                                collate_fn=safe_collate,
                                timeout=600 if num_workers > 0 else 0)   # stuck worker -> error, not endless wait

            def run(model, mods, meta):
                with torch.amp.autocast(device_type='cuda', enabled=(device.type == 'cuda')):
                    out = model(mods, meta)
                return torch.sigmoid(out.float()).cpu().numpy()

            acc, n_batches, t0 = {}, 0, time.time()
            it = iter(loader)
            with torch.no_grad():
                while True:
                    if time.time() > deadline:                       # stop before Kaggle kills us
                        print("DEADLINE reached: studies not done use metadata only")
                        break
                    try:
                        batch = next(it)
                    except StopIteration:
                        break
                    except Exception as e:
                        note("loader", e)
                        break
                    if batch is None:
                        continue
                    mods, meta, uids = batch
                    try:                                             # was outside any try before
                        mods = [m.to(device, dtype=torch.float32) for m in mods]
                        meta = meta.to(device, dtype=torch.float32)
                    except Exception as e:
                        note("to device", e)
                        continue
                    for model in models:
                        try:
                            p = run(model, mods, meta)
                            for u, row in zip(uids, p):
                                acc.setdefault(u, []).append(row)
                        except Exception as e:
                            note("model batch", e)
                            for i, u in enumerate(uids):             # retry one study at a time
                                try:
                                    p1 = run(model, [m[i:i+1] for m in mods], meta[i:i+1])
                                    acc.setdefault(u, []).append(p1[0])
                                except Exception as e2:
                                    note("model single", e2)
                    n_batches += 1
                    if n_batches % 10 == 0:
                        print(f"{len(acc)}/{len(good_uids)} studies | "
                              f"{(time.time() - t0) / max(len(acc), 1):.1f} sec per study")
            for i, u in enumerate(good_uids):
                if u in acc and len(acc[u]) > 0:
                    img_pred[i] = np.mean(acc[u], axis=0)
        except Exception as e:
            note("image models", e)
        print('End of image model predictions')

        # ---------- 5. combine with fallbacks
        has_img = ~np.isnan(img_pred).any(axis=1)
        final = np.full(img_pred.shape, 0.5)
        final[has_img] = img_pred[has_img]
        if meta_pred is not None:
            has_meta = ~np.isnan(meta_pred).any(axis=1)
            final[has_meta] = meta_pred[has_meta]          # metadata only
            both = has_img & has_meta
            if both.any():
                try:
                    final[both] = blend(img_pred[both], meta_pred[both], best_w)
                except Exception as e:
                    note("blend", e)
                    final[both] = best_w * img_pred[both] + (1 - best_w) * meta_pred[both]
        print(f"image ok: {int(has_img.sum())}/{len(good_uids)} | metadata ok: "
              f"{0 if meta_pred is None else len(good_uids)}/{len(good_uids)}")

        final = np.nan_to_num(final, nan=0.5)
        pred_df = pd.DataFrame(final, columns=TARGET_COLS)
        pred_df.insert(0, 'StudyInstanceUID', good_uids)
        if incomplete:
            fb = pd.DataFrame({'StudyInstanceUID': list(incomplete), **{c: 0.5 for c in TARGET_COLS}})
            pred_df = pd.concat([pred_df, fb], ignore_index=True)
        return pred_df

    try:
        pred_df = core()
    except Exception as e:
        note("core", e)
        pred_df = pd.DataFrame({'StudyInstanceUID': uids_all, **{c: 0.5 for c in TARGET_COLS}})

    try:
        return write(pred_df)
    except Exception as e:
        note("write", e)
        sub = pd.DataFrame({'StudyInstanceUID': uids_all, **{c: 0.5 for c in TARGET_COLS}})
        sub.to_csv(out_path, index=False)
        return sub

In [18]:
import glob

test_df = pd.read_csv('/kaggle/input/competitions/rsna-knee-abnormality-detection/test.csv')
test_series = pd.read_csv('/kaggle/input/competitions/rsna-knee-abnormality-detection/test_series.csv')
file_paths = glob.glob("/kaggle/input/competitions/rsna-knee-abnormality-detection/test_series/**/*.dcm", recursive=True)


    
print(f"Found {len(file_paths)} dicom files")


def get_dicom_metadata_df(file_paths):
    """
    Read metadata from a list of DICOM file paths
    and return a sorted DataFrame.
    """
    rows = []

    for path in file_paths:
        try:
            ds = pydicom.dcmread(path, stop_before_pixels=True)
        except Exception as e:
            print(f"Skipping bad file: {path} ({e})")
            continue

        row = {
            "file_path": path,
            "PatientID": tag(ds, "PatientID", None),
            "StudyInstanceUID": tag(ds, "StudyInstanceUID", None),
            "SeriesInstanceUID": tag(ds, "SeriesInstanceUID", None),
            "SOPInstanceUID": tag(ds, "SOPInstanceUID", None),

            "SeriesNumber": tag(ds, "SeriesNumber", None),
            "InstanceNumber": tag(ds, "InstanceNumber", None),
            "SequenceVariant": tag(ds, "SequenceVariant", None),
            "SeriesDescription": tag(ds, "SeriesDescription", None),
            "BodyPartExamined": tag(ds, "BodyPartExamined", None),
            "Laterality": tag(ds, "Laterality", None),
            "PatientSex": tag(ds, "PatientSex", None),
            "ScanningSequence": tag(ds, "ScanningSequence", None),
            "MRAcquisitionType": tag(ds, "MRAcquisitionType", None),
            "Rows": tag(ds, "Rows", None),
            "Columns": tag(ds, "Columns", None),
            "MagneticFieldStrength": tag(ds, "MagneticFieldStrength", None),
            "FlipAngle": tag(ds, "FlipAngle", None),
            "Manufacturer": tag(ds, "Manufacturer", None),
        }
        rows.append(row)

    df = pd.DataFrame(rows)

    # convert to numeric so sorting works correctly
    # df["InstanceNumber"] = pd.to_numeric(df["InstanceNumber"], errors="coerce")
    df["InstanceNumber"] = pd.to_numeric(df["InstanceNumber"].astype(str), errors="coerce")
    # sort: group by series, then order slices inside each series
    df = df.sort_values(
        by=["SeriesInstanceUID", "InstanceNumber", ]
    ).reset_index(drop=True)
    df.to_csv('metadata.csv', index=False)
    return df

def tag(ds, name, default=None):
    try:
        return getattr(ds, name, default)
    except Exception:
        return default
        
test_metadata = get_dicom_metadata_df(file_paths)


group_keys = ["StudyInstanceUID", "SeriesInstanceUID"]
value_cols = [col for col in test_metadata.columns if col not in group_keys]
# value_cols = [col for col in value_cols if col != "SequenceVariant"]
as_text = test_metadata.astype(str)      # text copy: lists and multi-values can't break nunique
constant_cols = [
    col for col in value_cols
    if col != "file_path"                # file_path is always a list
    and as_text.groupby(group_keys)[col].nunique().max() <= 1
]
# value_cols.append("SequenceVariant")
agg_dict = {
    col: ("first" if col in constant_cols else list) for col in value_cols
}

dicom_grouped = test_metadata.groupby(["StudyInstanceUID", "SeriesInstanceUID"], as_index=False).agg(agg_dict)
dicom_grouped

df_merged = pd.merge(
    test_series,
    dicom_grouped,
    on=["StudyInstanceUID", "SeriesInstanceUID"],
    how="left",
)

# 1. Define standardization mappings
laterality_map = {"R": "RIGHT", "L": "LEFT" , }

manufacturer_map = {
    # Siemens
    "SIEMENS": "Siemens",
    "Siemens Healthineers": "Siemens",
    "Siemens": "Siemens",
    # Philips
    "Philips Medical Systems": "Philips",
    "Philips Healthcare": "Philips",
    "Philips": "Philips",
    # GE (Optional but recommended)
    "GE MEDICAL SYSTEMS": "GE",
    "GEHC": "GE",
    # Canon / Toshiba / Fujifilm / Hitachi (Optional cleanup)
    "CANON_MEC": "Canon",
    "TOSHIBA": "Toshiba",
    "FUJIFILM Healthcare Corporation": "Fujifilm",
    "Hitachi Medical Corporation": "Hitachi",
}

# 2. Apply replacements to DataFrame
df_merged["Laterality"] = df_merged["Laterality"].replace(laterality_map)
df_merged["Manufacturer"] = df_merged["Manufacturer"].replace(manufacturer_map)
cols = [ 'Laterality', 'Manufacturer' ]
for i in cols:
    print(i, ':', df_merged[i].astype(str).unique())

df_merged.drop(columns=['SOPInstanceUID', 'SeriesNumber', 'PatientID'], inplace=True, errors='ignore')
def safe_max(x):
    if isinstance(x, list):
        vals = [v for v in x if pd.notna(v)]
        return max(vals) if vals else None
    return  x # was already NaN (no matching files)

df_merged['InstanceNumber'] = df_merged['InstanceNumber'].apply(safe_max)

df_merged.to_csv('test_merged.csv' , index = False)

Found 557 dicom files
Laterality : ['LEFT' 'None' '']
Manufacturer : ['Siemens' 'Toshiba' 'GE']


In [19]:


# meta_val, meta_gt = trainer.meta_val, trainer.meta_gt
# y_val, y_gt = trainer.y_val_bin, trainer.y_gt_bin
# test_merged = pd.read_csv('test_merged.csv')
# device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# import time, gc, psutil
# submission = submit_ensemble(test_df=test_df,
#                              test_series_df=test_merged,
#                              models= [combined_model, conv_model, dino_model],
#                              trainer=trainer, best_w=0.0, device=device,
#                              meta_ref= None, deadline=float('inf') )

# submission


# XAI

In [20]:
# ══════════════════════════════════════════════════════════════
#  XAI for MRI25DFusionModelViewRoutingDualEncoder
#  Needs: TARGET_COLS, SafeSubmissionDataset, safe_collate (from your submit code)
# ══════════════════════════════════════════════════════════════
import os
import numpy as np, pandas as pd, torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
from tqdm import tqdm

PLANES = ['Axial', 'Coronal', 'Sagittal']   # MUST match the modality order from KneeDataset
LABELS = TARGET_COLS


# ─────────────────────────────────────────────
# 1. Split the model forward in two parts
#    (so we can encode slices once, then hide slices cheaply)
# ─────────────────────────────────────────────
@torch.no_grad()
def encode_slices(model, mods):
    feats = []
    for s in mods:
        B, N, C, H, W = s.shape
        flat = s.reshape(B * N, C, H, W)
        fd = model.dino_slice_encoder(flat).view(B, N, -1)
        fc = model.conv_slice_encoder(flat).view(B, N, -1)
        feats.append(model.slice_fuse(torch.cat([fd, fc], dim=-1)))
    return feats                                           # list of (B, N, d)


def heads_from_feats(model, feats, masks=None):
    if masks is None:
        masks = [None] * len(feats)
    vecs    = [p(f, m) for f, p, m in zip(feats, model.slice_poolers, masks)]
    fused   = model.fusion(vecs)
    axial   = vecs[0]
    sc      = (vecs[1] + vecs[2]) / 2
    all_avg = torch.stack(vecs, dim=1).mean(dim=1)
    out = torch.cat([model.axial_head(torch.cat([axial,   fused], 1)),
                     model.sc_head(torch.cat([sc,         fused], 1)),
                     model.general_head(torch.cat([all_avg, fused], 1))], dim=1)
    return out[:, model.label_perm]                        # logits (B, 12)


# ─────────────────────────────────────────────
# 2. Deletion tests (slice level + plane level)
#    drop = logit(full) - logit(with part hidden)
#    drop > 0  → this part SUPPORTS the target
# ─────────────────────────────────────────────
@torch.no_grad()
def deletion_scores(model, mods):
    dev   = mods[0].device
    feats = encode_slices(model, mods)
    base  = heads_from_feats(model, feats)[0]              # (12,)
    P = len(feats)
    slice_drop, plane_drop = [], []
    for p in range(P):
        N = feats[p].shape[1]
        # one batch row per hidden slice
        exp   = [f.expand(N, -1, -1) for f in feats]
        masks = [None] * P
        masks[p] = torch.eye(N, dtype=torch.bool, device=dev)       # True = hidden
        out = heads_from_feats(model, exp, masks)                   # (N, 12)
        slice_drop.append((base - out).cpu().numpy())               # (N, 12)
        # hide the whole plane
        m_all = [None] * P
        m_all[p] = torch.ones(1, N, dtype=torch.bool, device=dev)
        out_p = heads_from_feats(model, feats, m_all)[0]
        plane_drop.append((base - out_p).cpu().numpy())             # (12,)
    return slice_drop, np.stack(plane_drop)                         # list[(N,12)], (P,12)


# ─────────────────────────────────────────────
# 3. Grad-CAM on both encoders (middle slice of every plane)
# ─────────────────────────────────────────────
class ActStore:
    """Grabs the last feature maps of both encoders during the forward pass."""
    def __init__(self, model):
        d = model.dino_slice_encoder.backbone
        c = model.conv_slice_encoder.backbone
        self.patch = d.config.patch_size
        self.dino, self.conv = [], []
        self.handles = [
            d.encoder.layer[-1].register_forward_pre_hook(self._dino, with_kwargs=True),
            c.stages[-1].register_forward_hook(self._conv),
        ]
    def _dino(self, mod, args, kwargs):
        self.dino.append(args[0] if args else kwargs['hidden_states'])   # (N, 1+tokens, C)
    def _conv(self, mod, inp, out):
        self.conv.append(out)                                            # (N, C, h, w)
    def close(self):
        for h in self.handles:
            h.remove()


def _up(cam, H, W):
    cam = cam[None, None].float()
    return F.interpolate(cam, size=(H, W), mode='bilinear', align_corners=False)[0, 0].detach().cpu().numpy()

def _cam_conv(act, grad):                                   # classic Grad-CAM
    w = grad.mean(dim=(1, 2), keepdim=True)
    return F.relu((w * act).sum(0))

def _cam_vit(act, grad, hh, ww):                            # gradient x activation per patch
    t = (act * grad).sum(-1)[1:]                            # drop CLS token
    assert t.numel() == hh * ww, f"token grid mismatch: {t.numel()} vs {hh}x{ww}"
    return F.relu(t).view(hh, ww)


def gradcam_all_labels(model, mods, slice_idx=None):
    """Returns CAMs (12, P, H, W) for 'dino', 'conv', 'both' + middle slice images."""
    model.eval()
    mods = [m.detach().clone().float().requires_grad_(True) for m in mods]  # needed: backbones may be frozen
    H, W = mods[0].shape[-2:]
    assert all(m.shape[-2:] == (H, W) for m in mods), "planes have different H/W"
    P    = len(mods)
    mids = [(m.shape[1] // 2 if slice_idx is None else slice_idx) for m in mods]

    store = ActStore(model)
    try:
        with torch.enable_grad():
            logits = model(mods)[0]                                         # (12,)
            assert len(store.dino) == P and len(store.conv) == P
            tensors = store.dino + store.conv
            n_lab   = logits.numel()
            cam = {e: np.zeros((n_lab, P, H, W), np.float32) for e in ('dino', 'conv')}
            for k in range(n_lab):
                grads = torch.autograd.grad(logits[k], tensors,
                                            retain_graph=(k < n_lab - 1), allow_unused=True)
                for p in range(P):
                    gd, gc, m = grads[p], grads[P + p], mids[p]
                    if gd is not None:
                        cam['dino'][k, p] = _up(_cam_vit(store.dino[p][m], gd[m],
                                                         H // store.patch, W // store.patch), H, W)
                    if gc is not None:
                        cam['conv'][k, p] = _up(_cam_conv(store.conv[p][m], gc[m]), H, W)
            probs = torch.sigmoid(logits).detach().cpu().numpy()
    finally:
        store.close()

    # scale per label across planes, so planes with no evidence stay dark
    for e in ('dino', 'conv'):
        cam[e] /= cam[e].max(axis=(1, 2, 3), keepdims=True) + 1e-8
    cam['both'] = 0.5 * (cam['dino'] + cam['conv'])
    cam['both'] /= cam['both'].max(axis=(1, 2, 3), keepdims=True) + 1e-8

    imgs = np.stack([mods[p][0, mids[p], 1].detach().cpu().numpy() for p in range(P)])  # centre channel
    return dict(cam=cam, imgs=imgs, prob=probs, mids=mids)


# ─────────────────────────────────────────────
# 4. Plots
# ─────────────────────────────────────────────
def plot_cam_grid(res, p, title, path, y_true=None, which='both'):
    img = res['imgs'][p]
    lo, hi = np.percentile(img, (1, 99))
    img = np.clip((img - lo) / (hi - lo + 1e-8), 0, 1)
    fig, axes = plt.subplots(3, 4, figsize=(16, 12))
    for k, ax in enumerate(axes.ravel()):
        ax.imshow(img, cmap='gray')
        ax.imshow(res['cam'][which][k, p], cmap='jet', alpha=0.45, vmin=0, vmax=1)
        t = f"{LABELS[k]}  p={res['prob'][k]:.2f}"
        if y_true is not None:
            t += f"  GT={y_true[k]:.2f}"
        ax.set_title(t, fontsize=11); ax.axis('off')
    fig.suptitle(title, fontsize=14)
    fig.tight_layout(); fig.savefig(path, dpi=110, bbox_inches='tight'); plt.close(fig)


def _heat(ax, M, xlabels, title, annotate=False, ylabels=True):
    v = np.abs(M).max() + 1e-8
    ax.imshow(M, cmap='RdBu_r', vmin=-v, vmax=v, aspect='auto')
    ax.set_title(title); ax.set_xticks(range(len(xlabels))); ax.set_xticklabels(xlabels, fontsize=8)
    ax.set_yticks(range(len(LABELS)))
    ax.set_yticklabels(LABELS if ylabels else [''] * len(LABELS), fontsize=9)
    if annotate:
        for i in range(M.shape[0]):
            for j in range(M.shape[1]):
                ax.text(j, i, f"{M[i, j]:.2f}", ha='center', va='center', fontsize=8)


def plot_importance(plane_drop, slice_drop, title, path):
    P = plane_drop.shape[0]
    fig, axes = plt.subplots(1, 1 + P, figsize=(5 + 5 * P, 6), gridspec_kw=dict(width_ratios=[1] + [2] * P))
    _heat(axes[0], plane_drop.T, PLANES, 'Plane deletion\n(logit drop)', annotate=True)
    for p in range(P):
        N = slice_drop[p].shape[0]
        _heat(axes[1 + p], slice_drop[p].T, list(range(N)), f'{PLANES[p]}: slice deletion', ylabels=False)
        axes[1 + p].set_xlabel('slice index')
    fig.suptitle(title + '   (red = supports the target, blue = works against it)', fontsize=13)
    fig.tight_layout(); fig.savefig(path, dpi=110, bbox_inches='tight'); plt.close(fig)


# ─────────────────────────────────────────────
# 5. One study → all explanations
# ─────────────────────────────────────────────
def explain_study(model, mods, uid, y_true=None, out_dir='xai_out', save_figs=True, check=False):
    dev  = next(model.parameters()).device
    mods = [m.to(dev, dtype=torch.float32) for m in mods]
    model.eval()

    if check:   # my split-forward must give the same output as the real model
        with torch.no_grad():
            a = model(mods); b = heads_from_feats(model, encode_slices(model, mods))
        assert torch.allclose(a, b, atol=1e-4), f"split forward differs: {(a-b).abs().max().item()}"
        print("sanity check OK: split forward == model forward")

    with torch.no_grad():
        slice_drop, plane_drop = deletion_scores(model, mods)
    res = gradcam_all_labels(model, mods)
    res['slice_drop'], res['plane_drop'] = slice_drop, plane_drop

    if save_figs:
        os.makedirs(out_dir, exist_ok=True)
        for p in range(len(mods)):
            plot_cam_grid(res, p, f"{uid} | {PLANES[p]} middle slice",
                          f"{out_dir}/{uid}_cam_{PLANES[p]}.png", y_true)
        plot_importance(plane_drop, slice_drop, uid, f"{out_dir}/{uid}_importance.png")
    return res


# ─────────────────────────────────────────────
# 6. Loop over many studies + average results
#    loader: batch_size=1, yields (mods, meta, uids)
# ─────────────────────────────────────────────
def run_xai(model, loader, gt_df=None, out_dir='xai_out', n_study_figs=5, max_studies=None):
    os.makedirs(out_dir, exist_ok=True)
    gt = gt_df.set_index('StudyInstanceUID')[LABELS] if gt_df is not None else None
    rec_plane, rec_prob = [], []
    slice_acc = [[] for _ in PLANES]
    cam_sum = img_sum = None
    n = 0

    for batch in tqdm(loader, total=max_studies or len(loader)):
        if batch is None:
            continue
        mods, meta, uids = batch
        assert mods[0].shape[0] == 1, "use batch_size=1"
        uid    = uids[0]
        y_true = gt.loc[uid].values if (gt is not None and uid in gt.index) else None
        try:
            res = explain_study(model, mods, uid, y_true, out_dir,
                                save_figs=(n < n_study_figs), check=(n == 0))
        except torch.cuda.OutOfMemoryError:
            print(f"OOM on {uid}, skipped"); torch.cuda.empty_cache(); continue

        for p, pl in enumerate(PLANES):
            for k, lb in enumerate(LABELS):
                rec_plane.append(dict(uid=uid, plane=pl, label=lb, drop=res['plane_drop'][p, k]))
            slice_acc[p].append(res['slice_drop'][p])
        rec_prob.append(dict(uid=uid, **{lb: res['prob'][k] for k, lb in enumerate(LABELS)},
                             **({f'GT_{lb}': y_true[k] for k, lb in enumerate(LABELS)} if y_true is not None else {})))

        cam_sum = res['cam']['both'] if cam_sum is None else cam_sum + res['cam']['both']
        img_sum = res['imgs']        if img_sum is None else img_sum + res['imgs']
        n += 1
        torch.cuda.empty_cache()
        if max_studies and n >= max_studies:
            break

    # ---- save tables
    df_plane = pd.DataFrame(rec_plane)
    df_plane.to_csv(f"{out_dir}/plane_importance_per_study.csv", index=False)
    pd.DataFrame(rec_prob).to_csv(f"{out_dir}/probs.csv", index=False)
    mean_plane = df_plane.pivot_table(index='label', columns='plane', values='drop', aggfunc='mean') \
                         .reindex(index=LABELS, columns=PLANES)
    mean_plane.to_csv(f"{out_dir}/plane_importance_mean.csv")
    print("\nMean plane importance (logit drop):\n", mean_plane.round(3))

    # ---- average plots
    mean_slice = [np.mean(np.stack(a), axis=0) for a in slice_acc]       # (N, 12) per plane
    plot_importance(mean_plane.values.T, mean_slice, f"MEAN over {n} studies", f"{out_dir}/MEAN_importance.png")
    mean_res = dict(cam={'both': cam_sum / n}, imgs=img_sum / n,
                    prob=np.array([np.mean([r[lb] for r in rec_prob]) for lb in LABELS]))
    for p in range(len(PLANES)):
        plot_cam_grid(mean_res, p, f"MEAN Grad-CAM over {n} studies | {PLANES[p]}",
                      f"{out_dir}/MEAN_cam_{PLANES[p]}.png")
    print(f"\nDone. {n} studies. Files in {out_dir}/")
    return mean_plane

In [21]:
from torch.utils.data import DataLoader


combined_model = MRI25DFusionModelViewRoutingDualEncoder(
            d=256, nhead=4, num_layers=2,
            num_modalities=3, max_slices=NUM_SLICES,
            num_metadata_features=33, d_meta=32
        )
path = '/kaggle/input/datasets/malihealavi/rsna-knee-model/best_model_checkpoint.pth'
ckpt = torch.load(path, map_location='cpu', weights_only=False) 
combined_model.load_state_dict(ckpt['model'])

combined_model = combined_model.to('cuda').eval()

gt_df = GT_VAL.copy()
ds     = SafeSubmissionDataset(gt_df, LIGHT_DF, meta_ref=None)
loader = DataLoader(ds, batch_size=1, shuffle=False, num_workers=0, collate_fn=safe_collate)

mean_plane = run_xai(combined_model, loader, gt_df, out_dir='xai_out', n_study_figs=5)

Loading weights:   0%|          | 0/223 [00:00<?, ?it/s]

Selected Encoder: dinov2_small
Selected Encoder: convnext_nano
Total Parameters: 14,952,560
Trainable Parameters: 14,952,560
Trainable Parameters: 0


  0%|          | 0/58 [00:00<?, ?it/s]

sanity check OK: split forward == model forward


100%|██████████| 58/58 [03:43<00:00,  3.86s/it]



Mean plane importance (logit drop):
 plane             Axial  Coronal  Sagittal
label                                     
ACL               0.241   -0.480     0.409
MCL               0.060   -0.212    -0.041
Medial Meniscus   0.169    0.372    -0.073
Lateral Meniscus  0.200    0.097    -0.217
Medial OA         0.149    0.506    -0.108
Lateral OA        0.111    0.411    -0.147
PF OA            -0.087    0.118    -0.054
Effusion          0.181    0.078     0.043
Synovitis         0.396    0.052    -0.001
Baker's           0.254   -0.131     0.095
Contusion         0.096   -0.394     0.394
Fracture          0.456   -0.374     0.440

Done. 58 studies. Files in xai_out/
